# Step 3 再構築からKWICへ — 踊り字の正規化・UniDic 解析・データセット再構築・KWICコンコーダンサ実装

> **用語の予習・復習**: `docs/glossary.md` の Step 3 を参照。キーワードを見て自分で説明してみてから読むこと。

## このステップの到達目標

1. 踊り字（`ゝゞヽヾ` とくの字点）を規則的に展開できる／`々` を展開しない理由を言える
2. UniDic の語形（surface / orthBase / lemma / lForm）の違いを説明できる
3. **解析辞書の選び方を，未知語率と平均語長の両方から論じられる**
4. 未知語率を指標に，前処理の良し悪しを判定できる／
   **その指標が何を測っているかが途中で変わりうることを説明できる**
5. 長さの偏りを吸収するチャンク分割ができる
6. KWICコンコーダンサの基本操作ができる

## 導入：踊り字はなぜ問題か

v1 コーパスの仮名踊り字の分布に留意すること。

| ファイル | ゝ | ゞ | ヽ | ヾ |
|---|---:|---:|---:|---:|
| 藤村『破戒』 | **218** | 0 | 0 | 0 |
| 岡本かの子『生々流転』 | 8 | **183** | 0 | 0 |
| 石川啄木『鳥影』 | 2 | 5 | **66** | 2 |
| 藤村『夜明け前』『家』『新生』『千曲川』 | **0** | 0 | 0 | 0 |

同じ藤村でも『破戒』だけ 218 箇所。これは作家の文体差ではなく，
**底本の正書法の差**である（『破戒』は新字旧仮名）。

正規化しないと，UniDic は `たゞ` を未知語として切り出す。すると

- `ただ`（副詞）が消え，`た` `ゞ` という無意味な列ができる
- 「藤村の文体は他と違う」という結論が，実は底本の違いを測っている

## `々` を展開してはいけない理由

`々` は UniDic に**語彙素として登録されている**（人々・時々・我々）。
これを `人人` に展開すると，かえって未知語になる。
**踊り字だからといって一律に扱わない。** 辞書の挙動を確かめてから決める。


In [ ]:
# ---- 共通の準備（毎回このセルから実行する）----------------------------
import os, sys, csv, json, math, random, shutil, subprocess, warnings
import importlib.util
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore', category=FutureWarning)

# リポジトリのルートを自動で探す（my_work/notebooks/ でも notebooks/ でも，上へたどる）
ROOT = Path.cwd()
while not (ROOT / 'config' / 'pipeline.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
print('ROOT =', ROOT)
if Path.cwd().resolve() == (ROOT / 'notebooks').resolve():
    print('[注意] 配布版（notebooks/）を直接開いている。実行すると次の git pull が止まる。\n'
          '       python scripts/copy_notebooks.py でコピーを作り，my_work/notebooks/ の方を開くこと。')

# 日本語フォント（□ にならないように）
for cand in ['Hiragino Sans', 'Yu Gothic', 'Meiryo',
             'Noto Sans CJK JP', 'IPAexGothic', 'MS Gothic']:
    if cand in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = cand
        break
else:
    print('[!] 日本語フォントが見つかりません。docs/00_setup_students.md §1.7 を参照。')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ---- 図はすべて SVG（ベクタ）で保存する ---------------------------------
# 論文・スライドに載せる図は拡大しても劣化してはならない。PNG は解像度が
# 固定されるので，投影や印刷で文字が潰れる。SVG なら任意の倍率で鮮明で，
# Illustrator / Inkscape で軸ラベルだけを直すこともできる。
FIG_EXT   = 'svg'
RASTER_DPI = 200          # rasterized=True の要素にだけ効く
plt.rcParams['svg.fonttype']       = 'path'   # 文字をアウトライン化して環境非依存に
plt.rcParams['savefig.transparent'] = False
# 画面へのインライン表示は既定（PNG）のままにする。
# InlineBackend.figure_formats を 'svg' に変えると，JupyterLab や
# VS Code の版によっては図がまったく表示されなくなることがある。
# **保存されるファイルは SVG** なので，論文・スライドに使うほうは
# ベクタで手元に残る。画面で拡大して見たいときは save_fig が表示する
# パスの .svg をブラウザで開くこと。

def need(path, hint=''):
    """必要な入力があるか確かめる。無ければ**理由を表示して** False を返す。

    セルを `if p.exists():` で囲むと，入力が無いときに何も起きない。
    受講生には「壊れている」と「まだ前の工程を実行していない」の区別が
    つかず，図が出ないという相談の大半がこれである。必ず理由を出す。
    """
    p = Path(path)
    try:
        ok = p.is_file() or (p.is_dir() and any(p.iterdir()))
    except OSError:
        ok = False
    if not ok:
        print(f'[未実行] {p} がありません。')
        if hint:
            print(f'         {hint}')
        print('         この Step の前のセルを上から順に実行すること。'
              '\n         それでも出ない場合は，前の Step のノートブックが'
              '最後まで通っているか確認する。')
    return ok


# 旧名 → 新名。**中身は 0–1 の割合なので per cent は誤称**である。
# 旧名の列を持つ古い出力も読めるように，読み替えを残す。
LEGACY_COLS = {'df_all_pct': 'df_all_prop', 'df_in_pct': 'df_in_prop'}


def read_table(path, **kw):
    """CSV を読み，**古い列名があれば新しい名前に読み替える**。

    列名の約束：割合（0–1）は ``_prop`` / ``_ratio`` / ``_share``，
    百分率（0–100）だけを ``_pct`` と綴る。``df_all_prop`` が 0.1584 なら
    15.84 % の意である。読み替えたときは黙らずに知らせる — 黙って直すと，
    手元の CSV と教材の列名が食い違っていることに気づけないため。
    """
    d = pd.read_csv(path, **kw)
    old = {k: v for k, v in LEGACY_COLS.items()
           if k in d.columns and v not in d.columns}
    if old:
        d = d.rename(columns=old)
        print('[note] 古い列名を読み替えた: '
              + '，'.join(f'{k}→{v}' for k, v in old.items())
              + '\n       07_descriptive_stats.py を実行し直すと'
                '新しい名前で書き出される。')
    return d


def load_meta(path=None, analysis_only=True):
    """メタデータを読む。既定では**分析に使う行だけ**を返す。

    除外するのは3種類。書誌としては残すが，集計に足してはいけない行である。
      superseded … v1 の合本。増補で分冊ごとに取り直したので，足すと
                   同じ作品を二重に数える
      merged     … 分冊。03b で canonical の巻に本文を統合したので，
                   この行はもう本文を持たない（『夜明け前』『家』）
      too_short  … 1チャンクにも満たず，チャンク単位の分析に乗らない

    生の表がほしいときは ``analysis_only=False``。
    """
    df = pd.read_csv(path or META)
    if analysis_only and 'completeness' in df.columns:
        drop = df['completeness'].isin(['superseded', 'merged', 'too_short'])
        if drop.any():
            names = '，'.join(df.loc[drop, 'title_aozora'].astype(str))
            print(f'[meta] 分析から除外 {int(drop.sum())} 行: {names}')
        df = df[~drop].reset_index(drop=True)
    return df


def w_ljust(text, width):
    """全角を2桁と数えて左詰めする。

    ``f'{s:<26}'`` は**文字数**で詰めるので，日本語の作品名を並べると
    桁が揃わない（全角は2桁ぶんの幅を占める）。表として読ませるなら
    表示幅で詰めること。

    **なお，一覧を出すなら ``show()`` で表にするほうがよい**（下記）。
    この関数は，表にしにくいもの（KWIC の前後文脈など）を print で
    並べるときに使う。
    """
    import unicodedata
    text = str(text)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in text)
    return text + ' ' * max(0, width - w)


# ---------------------------------------------------------------------------
# 分析結果の表示
# ---------------------------------------------------------------------------
# **一覧は print ではなく表で出す。**
#   * print は桁が揃わない（全角の幅）。数字の比較がしにくい
#   * 列に名前が付かないので，あとで見返したときに何の数字か分からない
#   * 並べ替えも絞り込みもできない
# 表にすると，列名がそのまま「何を測ったか」の記録になる。
# **ただし何でも表にするのではない。** 単発の数値・警告・KWIC の前後文脈は
# 文のほうが読みやすい。目安は「2列以上あるか」「行が並ぶか」。
TABLE_STYLES = [
    {'selector': 'caption',
     'props': [('caption-side', 'top'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '0 0 .4em 0'),
               ('color', '#33322e'), ('font-size', '95%')]},
    {'selector': 'th',
     'props': [('background', '#f2f2ef'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '.26em .7em'),
               ('border-bottom', '1px solid #c6c5bd'), ('white-space', 'nowrap')]},
    {'selector': 'td',
     'props': [('padding', '.22em .7em'), ('border-bottom', '1px solid #ecebe6')]},
    {'selector': 'tbody tr:hover td', 'props': [('background', '#f7f7f4')]},
]


def show(df, caption='', fmt=None, index=False, header=True, na='—', align=None):
    """DataFrame を表として表示する。Jupyter 以外でもエラーで止まらない。

    ``fmt`` は pandas の ``Styler.format`` に渡す辞書
    （例 ``{'一致率': '{:.1%}', 'G²': '{:.0f}'}``）。
    数値の列は自動で右寄せにする。``align`` で列ごとに寄せを指定できる。
    KWIC の左文脈を ``align={'左文脈': 'right'}`` にすると，
    **キーワードが縦に揃う**（等幅フォントに頼らずに揃う）。
    返り値は ``df`` なので ``t = show(df)`` として続けて使える。
    """
    if isinstance(df, pd.Series):
        df = df.to_frame()
    try:
        from IPython.display import display as _display
        st = df.style.format(fmt, na_rep=na) if fmt else df.style.format(na_rep=na)
        st = st.set_table_styles(TABLE_STYLES)
        num = list(df.select_dtypes('number').columns)
        if num:
            st = st.set_properties(subset=num, **{'text-align': 'right'})
        for col, side in (align or {}).items():
            if col in df.columns:
                st = st.set_properties(subset=[col],
                                       **{'text-align': side,
                                          'white-space': 'pre'})
        if caption:
            st = st.set_caption(caption)
        if not index:
            st = st.hide(axis='index')
        if not header:
            st = st.hide(axis='columns')
        _display(st)
    except Exception:                                   # noqa: BLE001
        # ノートブックの外（スクリプトから import したとき）でも読める形
        if caption:
            print(caption)
        print(df.to_string(index=index, header=header))
    return df


def grid(items, ncol=8, caption=''):
    """語の並びを ``ncol`` 列の表にして表示する。

    40 語を1行に流すと折り返しで読めない。列に切ると目で追える。
    順位が要るなら ``show()`` に順位列を付けた表を渡すこと。
    """
    items = [str(x) for x in items]
    rows = [items[i:i + ncol] for i in range(0, len(items), ncol)]
    rows = [r + [''] * (ncol - len(r)) for r in rows]
    t = pd.DataFrame(rows, columns=[f'_{i}' for i in range(ncol)])
    return show(t, caption=caption, header=False)


def work_rows(meta_df=None):
    """``work_stem`` からメタデータの行を引く辞書を作る。

    ``meta_df`` を省くと**分析対象外の行も含めた全件**から作る。
    表示用の名前は，分析から外した作品についても引けるほうがよい。

    **キーの綴りに注意。** 青空文庫の作品 ID は索引では 0 埋めされていない
    （``1743``）が，本パイプラインのファイル名は6桁に 0 埋めしてある
    （``000119_001743``）。素朴に連結すると ``000119_1743`` となり，
    **1件も一致しない**。辞書は空振りしても例外を出さないので，
    誰の何だか分からないまま最後まで通ってしまう。両方の綴りを登録する。

    ``file_v1`` は増補した作品では空である。``os.path.splitext(nan)`` は
    例外になるので，文字列であることを確かめてから使う。
    """
    if meta_df is None:
        meta_df = load_meta(analysis_only=False)
    d = {}
    for _, r in meta_df.iterrows():
        fv = r.get('file_v1')
        if isinstance(fv, str) and fv.strip():
            d[os.path.splitext(fv)[0]] = r
        pid = str(r.get('aozora_person_id') or '').strip()
        wid = str(r.get('aozora_work_id') or '').strip()
        if pid and wid and pid.lower() != 'nan' and wid.lower() != 'nan':
            for k in (f'{pid.zfill(6)}_{wid.zfill(6)}',
                      f'{pid}_{wid}', f'{pid.zfill(6)}_{wid}'):
                d[k] = r
    return d


def work_labels(meta_df=None, maxlen=12, with_year=False):
    """``work_stem`` → ``作者『作品』`` の対応表を返す。

    ``000119_001743`` と出されても誰の何だか分からない。距離の近い
    ペアを見るときに**どの作家のどの作品か**が分からなければ，
    「作家効果か時代効果か」という問いにそもそも答えられない。
    表示するときは必ずこれを通すこと。
    """
    out = {}
    for k, r in work_rows(meta_df).items():
        t = str(r.get('title_aozora') or '')
        lab = f"{r.get('author_ja', '?')}『{t[:maxlen]}』"
        if with_year and str(r.get('year_first') or '').strip():
            lab += f"({r['year_first']})"
        out[k] = lab
    return out


def attach_meta(df, cols, stem_col='work_stem', meta_df=None, quiet=False,
                fill_blank=True):
    """``df`` に足りないメタデータの列を，``work_stem`` から引いて補う。

    ``fill_blank=True``（既定）なら，**列はあるのに値が空**のセルも補う。
    列が無いより，列があって半分が空のほうが危ない。列が無ければ
    ``AttributeError`` で止まるが，値が空だと**図がそのまま描けてしまう**。
    たとえば突合が外れて多くの作品の ``year_first`` が空になっても，図は
    描けてしまい，それらが「初出年不明」の灰色で並ぶだけである。
    値の空きも数えて報告し，ここで補えるものは補う。

    分析スクリプトの出力は，その分析に要る列しか書かない。
    ``09_doc2vec.py`` の ``work_vectors.csv`` に ``genre_main`` が無いのは
    その一例である。ノートブックで ``wv.genre_main`` と書けば
    ``AttributeError: 'DataFrame' object has no attribute 'genre_main'``
    になるが，**足りないのは列であって情報ではない**。
    メタデータ表には必ずあるのだから，ここで引いて補えばよい。

    図の描画が出力 CSV の列構成に依存していると壊れやすい。分析スクリプトを
    書き換えるたびに図の描画がエラーで止まる。図の側で「要る列を宣言して取りに行く」
    ほうが，どちらを先に実行しても通る。

    引けなかった列は空のまま残し ``[warn]`` を出す。図の描画がエラーで止まるより，
    「この軸は色分けできなかった」と分かったうえで出るほうがよい。
    """
    df = df.copy()
    if stem_col not in df.columns:
        if not quiet:
            print(f'[warn] {stem_col} 列が無いので補完できない: {list(cols)}')
        for c in cols:
            if c not in df.columns:
                df[c] = ''
        return df

    rows = work_rows(meta_df)

    # **まずキーが合っているかを見る。** 合っていなければ何も補えない。
    # 「1件も合わない」のはたいてい 0 埋めの綴り違いで，黙って通すと
    # 全部の軸が空のまま図になる。
    stems = df[stem_col].astype(str)
    found = stems.map(lambda s: s in rows)
    if not quiet and not found.all():
        n_miss = int((~found).sum())
        lv = 'FATAL' if found.sum() == 0 else 'warn '
        print(f'[{lv}] {stem_col} がメタデータと突合できない行が '
              f'{n_miss}/{len(df)} 件ある: '
              + '，'.join(stems[~found].head(4)))
        if found.sum() == 0:
            print('        **1件も合っていない。** 作品 ID の 0 埋めの'
                  '綴り違いを疑うこと（例 000119_1743 と 000119_001743）。')
            print('        この表を作ったスクリプトのキーの作り方を直すこと。')

    def _blank(v):
        return v is None or str(v).strip().lower() in ('', 'nan', 'none')

    for c in cols:
        if c not in df.columns:
            vals = [(lambda r: '' if r is None or _blank(r.get(c))
                     else r.get(c))(rows.get(s)) for s in stems]
            df[c] = vals
            n = int(sum(1 for v in vals if str(v).strip()))
            if not quiet:
                mark = 'ok  ' if n == len(df) else 'warn'
                print(f'[{mark}] {c} をメタデータから補完: {n}/{len(df)} 件')
            continue

        if not fill_blank:
            continue
        # 列はある。空のセルだけを埋める。
        blank = df[c].map(_blank)
        if not blank.any():
            continue
        filled = 0
        vals = df[c].tolist()
        for i, (s, is_blank) in enumerate(zip(stems, blank)):
            if not is_blank:
                continue
            r = rows.get(s)
            if r is not None and not _blank(r.get(c)):
                vals[i] = r.get(c)
                filled += 1
        df[c] = vals
        if not quiet:
            left = int(sum(1 for v in vals if _blank(v)))
            mark = 'fix ' if left == 0 else 'warn'
            print(f'[{mark}] {c} は {int(blank.sum())}/{len(df)} 件が空だった'
                  f' → {filled} 件をメタデータから補完'
                  + ('' if left == 0 else f'（なお {left} 件が空）'))
            if left:
                print('        **その列で色分けする図・集計は，この件数を'
                      '報告に書くこと。**')
    return df


def label_points(ax, xs, ys, texts, fontsize=8, color='#333333', pad=4,
                 leader='line', leader_min=13, crowd_r=26,
                 leader_color='#8a8a83'):
    """散布図の注記を，重ならない位置だけに置き，遠いものは引き出し線で結ぶ。

    素朴に ``ax.annotate(t, (x, y))`` と書くと，**注目すべき点ほど一箇所に
    固まる**ので注記が必ず重なって読めなくなる。文語標識の上位は
    どれも口語標識がほぼ 0 で，対数軸の右下隅に密集するのが典型である。

    そこで点の周囲を順に試し，他の注記とも他の点とも重ならず，かつ軸の
    内側に収まる位置があればそこに置く。どこにも置けない注記は**置かずに
    数だけ報告する**。読めない字を重ねるより，図の外（下の表）で番号から
    引くほうがよい。

    **離れた位置に置いた注記は，引き出し線で点と結ぶ。** 避けた結果として
    注記は点から離れるので，線が無いとどの点の名前なのか分からなくなる
    ——密集した領域では隣の点の名前だと読まれる。線があれば，遠くへ逃がす
    ことに副作用が無くなるので，**近くに空きが無い注記も置ける**ようになる
    （候補の輪を 24・30 ポイントまで広げてあるのはそのため）。

    ``leader``
        ``'line'``（既定）… 矢じりの無い細線で結ぶ。図版の慣例はこちら。
        6.5pt の文字に矢じりを付けると，マーカーそのものを覆って点が読めなくなる
        ``'arrow'`` … 小さな矢じりを付ける
        ``'none'`` … 結ばない（従来どおり）
    ``leader_min``
        この距離（ポイント）より遠くに置いた注記を結ぶ。既定は 0，
        つまり**すべて結ぶ**。注記は必ず点から離れた位置に置かれるので，
        離れている以上「どの点の名前か」は線でしか確定しない。
        線を省くと，隣の点の名前だと読まれる余地が残る。
    ``crowd_r``
        注記の近くに**自分以外の点**がこの半径（ピクセル）内にあるかを
        見る。``leader_min`` を上げて線を減らしたときでも，
        紛れる相手が居る注記だけは必ず結ぶための保険である。

    表示座標で矩形の重なりを見るので，**軸の位置が確定してから**呼ぶ。
    ``fig.tight_layout()`` はこの関数より**前**に呼ぶこと（後で呼ぶと軸が
    動き，せっかく避けた位置がずれる）。戻り値は置けた注記の数。
    """
    from matplotlib.transforms import Bbox
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    texts = list(texts)
    if len(texts) == 0:
        return 0
    # **長さが違えば黙って切り詰めずに止める。** zip は短いほうに合わせるので，
    # 座標だけを絞り込んで名前を絞り忘れると，先頭から順に**別の作品の名前**が
    # 貼られた図が，何の警告も出さずに出来上がる。これがいちばん重い事故である。
    if not (len(xs) == len(ys) == len(texts)):
        raise ValueError(
            f'label_points: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'名前={len(texts)}）。座標と名前を同じ添字で絞り込むこと。'
            'たとえば P[pick,0] と組むのは names[pick] であって names ではない。')
    fig = ax.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    axbb = ax.get_window_extent(renderer=ren)

    def pad_box(b, w=2.0, h=1.5):
        """注記の矩形に**絶対量の余白**を足す。

        倍率（``expanded(1.08, …)``）では足りない。1桁の数字は幅 8px ほど
        なので 8% は 0.6px にしかならず，隣り合う注記が触れるほど近くても
        「重なっていない」と判定される。**``21`` が「21」と読める**のは
        これが原因である。文字の大小によらず一定の余白を確保する。
        """
        return Bbox.from_extents(b.x0 - w, b.y0 - h, b.x1 + w, b.y1 + h)

    def bb_of(ann):
        # Annotation 自身の get_window_extent を使うこと。
        # Text.get_window_extent(ann, ...) を呼ぶと xy の位置が無視され，
        # xytext のオフセットを絶対座標と見た矩形が返って判定が壊れる。
        return ann.get_window_extent(renderer=ren)

    blocked = []
    for coll in ax.collections:
        try:
            for p in coll.get_offsets():
                px, py = ax.transData.transform(p)
                blocked.append(Bbox.from_bounds(px - pad, py - pad,
                                                2 * pad, 2 * pad))
        except Exception:                                    # noqa: BLE001
            pass

    # **まっすぐ真上・真下を先に試す。** 点の直上に中央揃えで置ければ，
    # それがいちばん素直で，引き出し線も要らない。横へずらすのは，
    # 直上が塞がっていたときの次善である。
    #
    # 横へずらす輪は 12 ポイントから始める。**線が線として見える長さを
    # 確保する**ため。8 ポイントに置くと引き出し線が3ピクセルの点にしか
    # ならず，汚れと区別がつかない。
    # **真上に置けなければ，まず真上へ逃がす。** 横へ逃がすと注記の左右の
    # 順序が点の順序と入れ替わり，引き出し線も交差する。真上に段を重ねる
    # 限り，x は動かないので順序は必ず保たれる。横へずらすのは最後。
    # **横のずらし幅は小さく取る。** 横へ 30 ポイントも動かすと，注記が
    # 隣の点の真上に乗り，引き出し線で結んでも読みにくい。真上に段を
    # 重ねるほうが先で（x が動かないので順序が保たれる），横は 8→18
    # ポイントの範囲に収める。
    CAND = [(0, 9), (0, -11), (0, 20), (0, -22), (0, 31), (0, -33),
            (8, 5), (-8, 5), (8, -12), (-8, -12),
            (11, 0), (-11, 0),
            (13, 9), (-13, 9), (13, -16), (-13, -16),
            (18, 0), (-18, 0), (18, 14), (-18, 14),
            (0, 42), (0, -44)]

    def _ha(dx):
        # dx が 0 なら**中央揃え**。ここを 'left' にすると，真上に置いた
        # つもりの注記が文字幅の半分だけ右にずれ，隣の点の上に乗る。
        return 'center' if dx == 0 else ('left' if dx > 0 else 'right')
    placed, chosen, skipped = [], [], 0
    for x, y, t in zip(xs, ys, texts):
        # **自分が指している点は避けない。** 除かないと，注記は必ず
        # 自分の点の隣に来るので全部「重なる」と判定され，1つも置けない。
        ox, oy = ax.transData.transform((x, y))
        near = [b for b in blocked
                if not (abs((b.x0 + b.x1) / 2 - ox) < 1
                        and abs((b.y0 + b.y1) / 2 - oy) < 1)]
        for dx, dy in CAND:
            ann = ax.annotate(str(t), (x, y), textcoords='offset points',
                              xytext=(dx, dy), fontsize=fontsize, color=color,
                              ha=_ha(dx),
                              va='bottom' if dy >= 0 else 'top', zorder=6)
            bb = pad_box(bb_of(ann))
            inside = (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                      and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1)
            if inside and not any(bb.overlaps(b) for b in placed + near):
                placed.append(bb)
                chosen.append((ann, x, y, str(t), dx, dy))
                break
            ann.remove()
        else:
            skipped += 1

    # ---- 交差をほどく ----------------------------------------------------
    # **引き出し線が交差すると，注記の左右の順序が点の順序と入れ替わる。**
    # 文語標識の上位のように順位そのものが意味を持つ図では，2 と 3 が
    # 入れ替わって並ぶだけで読み違えられる。交差している2件を見つけ，
    # **位置を入れ替えて交差が解ければ入れ替える**（2-opt）。
    def _cross(p, q, r, s):
        def o(a, b, c):
            return ((b[0] - a[0]) * (c[1] - a[1])
                    - (b[1] - a[1]) * (c[0] - a[0]))
        return (((o(r, s, p) > 0) != (o(r, s, q) > 0))
                and ((o(p, q, r) > 0) != (o(p, q, s) > 0)))

    kpt = fig.dpi / 72.0

    def _seg(i):
        ann, x, y, t, dx, dy = chosen[i]
        ox, oy = ax.transData.transform((x, y))
        return (ox, oy), (ox + dx * kpt, oy + dy * kpt)

    def _set_off(i, dx, dy):
        ann, x, y, t, _, _ = chosen[i]
        ann.set_position((dx, dy))
        ann.set_ha(_ha(dx))
        ann.set_va('bottom' if dy >= 0 else 'top')
        chosen[i] = (ann, x, y, t, dx, dy)

    def _fits(i, bb):
        ox, oy = ax.transData.transform((chosen[i][1], chosen[i][2]))
        if not (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1):
            return False
        return not any(bb.overlaps(b) for b in blocked
                       if abs((b.x0 + b.x1) / 2 - ox) > 1
                       or abs((b.y0 + b.y1) / 2 - oy) > 1)

    swaps = 0
    for _ in range(3):
        improved = False
        for i in range(len(chosen)):
            for j in range(i + 1, len(chosen)):
                if not _cross(*_seg(i), *_seg(j)):
                    continue
                di, dj = chosen[i][4:6], chosen[j][4:6]
                _set_off(i, *dj)
                _set_off(j, *di)
                bi = pad_box(bb_of(chosen[i][0]))
                bj = pad_box(bb_of(chosen[j][0]))
                others = [b for k2, b in enumerate(placed) if k2 not in (i, j)]
                good = (not bi.overlaps(bj)
                        and not any(bi.overlaps(b) or bj.overlaps(b)
                                    for b in others)
                        and _fits(i, bi) and _fits(j, bj)
                        and not _cross(*_seg(i), *_seg(j)))
                if good:
                    placed[i], placed[j] = bi, bj
                    swaps += 1
                    improved = True
                    continue
                _set_off(i, *di)
                _set_off(j, *dj)

                # 入れ替えが収まらないときは，**片方を別の候補位置へ動かす**。
                # 入れ替えは2つの箱の大きさが違うと失敗しやすい（数字1桁と
                # 作者名では幅が違う）。動かすほうは箱の大きさが変わらない。
                moved = False
                for who, other in ((i, j), (j, i)):
                    d0 = chosen[who][4:6]
                    for cx, cy in CAND:
                        if (cx, cy) == tuple(d0):
                            continue
                        _set_off(who, cx, cy)
                        bw = pad_box(bb_of(chosen[who][0]))
                        rest = [b for k2, b in enumerate(placed) if k2 != who]
                        if (_fits(who, bw)
                                and not any(bw.overlaps(b) for b in rest)
                                and not _cross(*_seg(who), *_seg(other))
                                and not any(_cross(*_seg(who), *_seg(k2))
                                            for k2 in range(len(chosen))
                                            if k2 != who)):
                            placed[who] = bw
                            swaps += 1
                            moved = improved = True
                            break
                        _set_off(who, *d0)
                    if moved:
                        break
        if not improved:
            break

    # ---- 引き出し線 ------------------------------------------------------
    # **線は配置が全部決まってから付ける。** arrowprops を付けた
    # Annotation の get_window_extent は「文字＋線」の外接矩形を返すので，
    # 配置の判定に使うと自分の点と必ず重なり，1件も置けなくなる。
    n_leader = 0
    if leader in ('line', 'arrow'):
        style = '-' if leader == 'line' else '-|>'
        for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
            ox, oy = ax.transData.transform((x, y))

            def dist_to_box(px, py, b=bb):
                # 文字の矩形から点までの距離。矩形の中なら 0。
                ddx = max(b.x0 - px, 0, px - b.x1)
                ddy = max(b.y0 - py, 0, py - b.y1)
                return (ddx * ddx + ddy * ddy) ** .5

            # **素直に置けたものには線を引かない。** 点の直上（または直下）に
            # 中央揃えで載っていて，しかもその注記にいちばん近い点が自分の
            # 点であれば，どの点の名前かは見れば分かる。線はかえって邪魔
            # である。横へ逃がしたものだけを結ぶ。
            if dx == 0 and abs(dy) <= 12:
                continue            # 点の真上・真下の一段目 → 線は要らない
            # それ以外は結ぶ。**段を上げたものも結ぶ。** 一段上げた注記の
            # 真下には別の点の注記が入るので，どちらの点のものか分からなく
            # なる。横へずらしたものは言うまでもない。
            d_other = min(
                (dist_to_box((b.x0 + b.x1) / 2, (b.y0 + b.y1) / 2)
                 for b in blocked
                 if abs((b.x0 + b.x1) / 2 - ox) > 1
                 or abs((b.y0 + b.y1) / 2 - oy) > 1),
                default=float('inf'))
            if (dx * dx + dy * dy) ** .5 < leader_min and d_other >= crowd_r:
                continue
            ann.remove()
            ax.annotate(t, (x, y), textcoords='offset points',
                        xytext=(dx, dy), fontsize=fontsize, color=color,
                        ha=_ha(dx),
                        va='bottom' if dy >= 0 else 'top', zorder=6,
                        arrowprops=dict(arrowstyle=style, linewidth=.55,
                                        color=leader_color, alpha=.9,
                                        shrinkA=1.5, shrinkB=2.5,
                                        mutation_scale=7))
            n_leader += 1

    # ---- 誤読の自己点検 --------------------------------------------------
    # **注記の最寄りの点が自分の点でないものを数える。** これが
    # 「ラベルとデータ点がずれて見える」の正体である。引き出し線を
    # 引いてあれば誤読にはならないが，線を切った設定では危険なので，
    # そのときだけ警告を出す。
    risky = []
    for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
        ox, oy = ax.transData.transform((x, y))
        cx, cy = (bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2
        d_own = ((cx - ox) ** 2 + (cy - oy) ** 2) ** .5
        d_other = min(
            (((b.x0 + b.x1) / 2 - cx) ** 2 + ((b.y0 + b.y1) / 2 - cy) ** 2) ** .5
            for b in blocked
            if abs((b.x0 + b.x1) / 2 - ox) > 1 or abs((b.y0 + b.y1) / 2 - oy) > 1
        ) if len(blocked) > 1 else float('inf')
        if d_other < d_own:
            risky.append(t)
    left = sum(1 for i in range(len(chosen)) for j in range(i + 1, len(chosen))
               if _cross(*_seg(i), *_seg(j)))
    if skipped:
        print(f'[fig] 重なるため {skipped} 件の注記を省いた（表で引くこと）')
    if left:
        print(f'[warn] 引き出し線の交差が {left} 件ほどけなかった。'
              '注記の左右の順序が点の順序と食い違う。'
              '注記を短くするか，件数を減らすこと。')
    if risky:
        head = '，'.join(str(r) for r in risky[:6])
        more = f' ほか{len(risky) - 6}件' if len(risky) > 6 else ''
        if leader in ('line', 'arrow'):
            print(f'[fig] {len(risky)} 件の注記は別の点のほうが近い'
                  f'（{head}{more}）。引き出し線で結んであるので読み違えない。')
        else:
            print(f'[warn] {len(risky)} 件の注記は**別の点のほうが近い**'
                  f'（{head}{more}）。leader="none" では読み違えが起きる。')
    return len(placed)
def _proj_versions():
    """射影に関わる版を並べる（うまくいかないときの手がかり）。"""
    import importlib
    out = []
    for nm in ['numpy', 'numba', 'llvmlite', 'pynndescent', 'sklearn']:
        try:
            out.append(f'{nm} ' + str(getattr(importlib.import_module(nm),
                                              '__version__', '?')))
        except Exception:                                    # noqa: BLE001
            out.append(f'{nm} ×')
    return '／'.join(out)

def umap_diagnosis(e):
    """UMAP が使えないときに，**何をすればよいか**を出す。"""
    import sys
    print(f'[NG  ] UMAP が使えない: {type(e).__name__}: {e}')
    print(f'       このカーネルの Python = {sys.executable}')
    print(f'       {_proj_versions()}')
    if isinstance(e, ModuleNotFoundError):
        # **入れた先とカーネルの環境が違う**のが圧倒的に多い。
        # uv add は「プロジェクト（pyproject.toml のある場所）」単位なので，
        # dh_project/pyproject.toml が無い，または dh_project の外に clone
        # した場合は，uv は別のプロジェクトに入れる。カーネルの .venv には入らない。
        print('       **この環境には入っていない。** 入れた先が違う可能性が高い')
        print('       （uv add はプロジェクト単位。~/Documents/dh_project に')
        print('        pyproject.toml が無いと，別のプロジェクトに入る）。')
        print('       この環境を名指しして入れるのが確実:')
        import platform as _pf
        if sys.platform == 'darwin' and _pf.machine() == 'x86_64':
            # **Intel Mac は版を固定する。** llvmlite の x86_64 wheel は
            # 0.45.1 が最後で，0.46 以降は arm64 のみ。固定しないと
            # ソースからのビルドに切り替わり，Homebrew の LLVM と版が合わずに
            # 失敗する（llvmlite 0.49 は LLVM 22 を要求）。
            print('       （Intel Mac なので**版を固定する**。'
                  'llvmlite の x86_64 wheel は 0.45.1 が最後）')
            print(f'         uv pip install --python "{sys.executable}" \\')
            print('             --only-binary :all: \\')
            print('             "numba==0.62.1" "llvmlite==0.45.1" '
                  '"numpy<2.4" umap-learn')
        else:
            print(f'         uv pip install --python "{sys.executable}" umap-learn')
        print('       入れたら**カーネルを再起動**して，このセルから実行し直す。')
    else:
        print('       import は通るが使えない，という種類の失敗である'
              '（別パッケージの umap／numba と numpy の版違い／'
              'numba のキャッシュ）。')
    print('       切り分けの全項目:')
    print('         import sys, subprocess; print(subprocess.run('
          '[sys.executable,')
    print("             str(ROOT/'scripts'/'check_umap.py')], "
          'capture_output=True,')
    print('             text=True).stdout)')

def project(Xn, how='umap', seed=20260920, n_neighbors=15, min_dist=0.12,
            perplexity=30):
    """高次元の行列を2次元に圧縮して射影する。**どの方法で射影したかを必ず返す。**

    ``how`` は ``'umap'``／``'tsne'``／``'auto'``。既定の ``'umap'`` は，
    使えなければ**止まって理由を出す**。``'auto'`` のときだけ t-SNE に切り替わる。
    **黙って別の方法に替えないのが肝心である**（図は出るが塊の見え方は
    変わるので，環境の問題を分析結果と読み違える）。

    Step 5 の §3（主成分分析との比較）と §4（ギャラクシー）が共有する。
    """
    import importlib
    Xn = np.asarray(Xn, dtype=np.float32)
    if how in ('auto', 'umap'):
        try:
            m = importlib.import_module('umap')
            if not hasattr(m, 'UMAP'):
                # PyPI には umap（別物）と umap-learn（本物）がある。
                # pip install umap をしていると import umap はそちらを拾う。
                raise ImportError(
                    f'umap に UMAP クラスが無い（{getattr(m, "__file__", "?")}）。'
                    '別パッケージの umap が入っている。'
                    'umap を外して umap-learn を入れること')
            P = m.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                       metric='cosine', random_state=seed).fit_transform(Xn)
            return (np.asarray(P, dtype=np.float32),
                    f'UMAP {getattr(m, "__version__", "")}'
                    f' (n_neighbors={n_neighbors}, min_dist={min_dist}, cosine)')
        except Exception as e:                               # noqa: BLE001
            umap_diagnosis(e)
            if how == 'umap':
                # **黙って別の方法に替えない。** どうしても t-SNE で
                # 進めたいときは 'tsne' と明示し，報告にもそう書くこと。
                raise
            print('[warn] how="auto" なので t-SNE に切り替える。'
                  '**図と報告に t-SNE と書くこと。**')
    from sklearn.manifold import TSNE
    P = TSNE(n_components=2, perplexity=perplexity, metric='cosine',
             init='pca', random_state=seed).fit_transform(Xn)
    return (np.asarray(P, dtype=np.float32),
            f't-SNE (perplexity={perplexity}, cosine)')


def proj_quality(Xn, P, k=10):
    """射影が原空間の近さをどれだけ歪めているかを3つの数で返す。

    ``trust``  … 2次元で近く見える点が原空間でも近いか（局所・1が最良）
    ``keep``   … 原空間の上位 k 近傍のうち画面でも上位 k に入る語数
    ``rho``    … 原空間の距離と画面の距離の順位相関（**大域**の保存）

    局所（trust・keep）と大域（rho）は別物である。**UMAP は局所に強く，
    主成分分析は大域に強い**——これを目で見ずに数で確かめるための関数。
    """
    from scipy.spatial.distance import pdist
    from scipy.stats import spearmanr
    from sklearn.manifold import trustworthiness
    Xn, P = np.asarray(Xn, np.float32), np.asarray(P, np.float32)
    S = Xn @ Xn.T
    np.fill_diagonal(S, -np.inf)
    nn_t = np.argsort(-S, axis=1)[:, :k]
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    nn_p = np.argsort(d2, axis=1)[:, :k]
    keep = np.array([len(set(a) & set(b)) for a, b in zip(nn_t, nn_p)])
    trust = float(trustworthiness(Xn, P, n_neighbors=k, metric='cosine'))
    rho = float(spearmanr(pdist(Xn, 'cosine'), pdist(P))[0])
    return {'trust': trust, 'keep': keep, 'rho': rho}


def reserve_right(fig, frac=0.80):
    """面の外に凡例を置いた図で，**右に余白を確保する**。

    ``tight_layout()`` は面の外に置いた凡例を数えないので，そのままだと
    凡例が図の枠から出る。静止版は ``bbox_inches='tight'`` で枠内に収まるが，
    **HTML に埋め込む版は切り取らない**（切り取ると点の位置の割合が
    ずれる）ので，凡例が切れて読めなくなる。

    ``tight_layout()`` の**後**，注記（``label_points``）の**前**に呼ぶ。
    """
    fig.subplots_adjust(right=frac)


def save_fig(fig, stem, out=None):
    """図を SVG で保存してパスを表示する。

    stem は拡張子なしの名前（例 'Step1_period_balance'）。
    点が数千個ある散布図は，散布図だけ rasterized=True にしておくと
    軸と文字はベクタのままファイルが軽くなる。
    """
    d = Path(out) if out else OUT
    d.mkdir(parents=True, exist_ok=True)
    path = d / f'{stem}.{FIG_EXT}'
    fig.savefig(path, format=FIG_EXT, dpi=RASTER_DPI, bbox_inches='tight')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB)')
    return path


# ---------------------------------------------------------------------------
# 対話的な図（SVG はそのまま残す）
# ---------------------------------------------------------------------------
# 散布図の点が何百個あると，注記を付けられるのはごく一部である。残りの点は
# 「どの語か」が分からないまま眺めることになる。かといって全点に名前を
# 付ければ図は読めない。
#
# そこで**同じ図から2つ出す**。
#   * ``<stem>.svg``  … 論文・配布用。加筆も拡大も自由
#   * ``<stem>.html`` … 授業・探索用。SVG をそのまま埋め込み，
#                       その上にポインタの判定範囲を重ねて，指した点の語を出す
#
# **HTML は SVG を作り直さない。同じ SVG を中に入れる。** 別に描き直すと
# 図が2種類できて，どちらが正かが分からなくなる。注記（bursty な語の
# ラベル）も SVG の中にあるのでそのまま残る。
#
# 外部の JS ライブラリは使わない。CDN が塞がれたマシンでも開けるようにする。
INTERACTIVE_CSS = """
:root { --ink:#1f1e1b; --ink2:#5a5a55; --line:#d8d7d0; --surface:#ffffff;
        --wash:#f7f7f4; --accent:#184f95; }
* { box-sizing:border-box; }
body { margin:0; padding:24px 16px 48px; background:var(--wash);
       color:var(--ink); font-family:"Hiragino Sans","Noto Sans JP",
       "Yu Gothic",system-ui,sans-serif; line-height:1.6; }
.wrap { max-width:1100px; margin:0 auto; }
h1 { font-size:1.15rem; margin:0 0 .2em; font-weight:650; }
.sub { color:var(--ink2); font-size:.86rem; margin:0 0 1.1em; }
.card { background:var(--surface); border:1px solid var(--line);
        border-radius:10px; padding:14px; }
.figbox { position:relative; }
.figbox svg { width:100%; height:auto; display:block; }
#hit { position:absolute; inset:0; cursor:crosshair; }
#ring { position:absolute; width:22px; height:22px; margin:-11px 0 0 -11px;
        border:2px solid var(--accent); border-radius:50%;
        pointer-events:none; opacity:0; transition:opacity .08s; }
#tip { position:absolute; z-index:5; min-width:190px; max-width:290px;
       background:var(--surface); border:1px solid var(--line);
       border-radius:8px; box-shadow:0 6px 20px rgba(0,0,0,.13);
       padding:9px 11px; font-size:.8rem; pointer-events:none; opacity:0;
       transition:opacity .08s; }
#tip .term { font-size:1.05rem; font-weight:650; letter-spacing:.02em;
             margin-bottom:.35em; word-break:break-all; }
#tip dl { display:grid; grid-template-columns:auto 1fr; gap:1px 10px;
          margin:0; }
#tip dt { color:var(--ink2); font-size:.74rem; white-space:nowrap; }
#tip dd { margin:0; text-align:right; font-variant-numeric:tabular-nums;
          font-weight:600; }
.bar { display:flex; gap:10px; align-items:center; flex-wrap:wrap;
       margin:14px 0 0; font-size:.82rem; color:var(--ink2); }
.bar input { font:inherit; padding:5px 9px; border:1px solid var(--line);
             border-radius:6px; min-width:190px; background:var(--surface); }
.bar a { color:var(--accent); }
table { border-collapse:collapse; width:100%; font-size:.78rem;
        margin-top:10px; }
th,td { padding:4px 8px; border-bottom:1px solid #ecebe6; text-align:left;
        white-space:nowrap; }
th { background:var(--wash); position:sticky; top:0; font-weight:650; }
td.num { text-align:right; font-variant-numeric:tabular-nums; }
tbody tr:hover td { background:var(--wash); }
tbody tr.on td { background:#eaf1fb; }
.scroll { max-height:340px; overflow:auto; border:1px solid var(--line);
          border-radius:8px; margin-top:10px; }
.hint { font-size:.78rem; color:var(--ink2); margin:.6em 0 0; }
#links { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#links line { stroke:var(--accent); stroke-width:1.1; opacity:.55; }
#links circle { fill:none; stroke:var(--accent); stroke-width:1.4; opacity:.8; }
#marks { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#marks circle { fill:none; stroke:#d55e00; stroke-width:1.6; opacity:.9; }
#tip .notes { margin:.45em 0 0; font-size:.76rem; color:var(--ink);
              border-top:1px solid var(--line); padding-top:.4em;
              line-height:1.5; word-break:break-all; }
#tip .notes b { color:var(--ink2); font-weight:600; }
.prov { font-size:.72rem; color:var(--ink2); margin:.9em 0 0;
        border-top:1px solid var(--line); padding-top:.6em;
        font-variant-numeric:tabular-nums; }
.danger { background:#fdf0ea; border:1px solid #e8a37c; border-radius:8px;
          padding:9px 12px; font-size:.85rem; color:#8a3b10;
          margin:0 0 12px; }
"""

INTERACTIVE_JS = r"""
// 点は data-* ではなく JSON で渡す。語はコーパス由来の任意の文字列なので，
// **HTML に文字列連結で差し込まない**（textContent で入れる）。
// 見出し（keys・nhead）は全点で同じなら1回だけ入っている。点が1万個ある
// 図では，これで HTML が 1 MB 以上軽くなる。古い形（配列だけ）も読む。
const RAW = JSON.parse(document.getElementById('pts-data').textContent);
const PTS = Array.isArray(RAW) ? RAW : RAW.pts;
const KEYS = (RAW && RAW.keys) || [];
const NHEAD = (RAW && RAW.nhead) || '';
const LINKNOTES = !!(RAW && RAW.linknotes);
function pairsOf(p) {
  if (p.fields) return p.fields;
  if (p.v) return p.v.map((x, i) => [KEYS[i] || '', x]);
  return [];
}
function notesOf(p) {
  if (p.notes) return p.notes;
  if (p.n) return [NHEAD, p.n];
  // 本文が無く linknotes が立っているときは，線で結ぶ先の語を並べる
  if (LINKNOTES && p.links && p.links.length) {
    return [NHEAD, p.links.map(j => (PTS[j] || {}).term || '').join(' ')];
  }
  return null;
}
const box = document.getElementById('hit');
const tip = document.getElementById('tip');
const ring = document.getElementById('ring');
const rows = Array.from(document.querySelectorAll('tbody tr'));
const links = document.getElementById('links');
const marks = document.getElementById('marks');

// **最も近い点を拾う。** 点の直径は数ピクセルしかないので，
// 「真上に置く」ことを要求すると誰も当てられない（dataviz の規則）。
// カーソルに最も近い点を選び，遠すぎるときだけ何も出さない。
function nearest(px, py, w, h) {
  let best = null, bd = 1e9;
  for (const p of PTS) {
    const dx = p.x * w - px, dy = p.y * h - py;
    const d = dx * dx + dy * dy;
    if (d < bd) { bd = d; best = p; }
  }
  return Math.sqrt(bd) <= 34 ? best : null;   // 34px より遠ければ出さない
}

function fill(p) {
  tip.textContent = '';
  const h = document.createElement('div');
  h.className = 'term';
  h.textContent = p.term;                     // ← 連結しない
  tip.appendChild(h);
  const dl = document.createElement('dl');
  for (const [k, v] of pairsOf(p)) {
    const dt = document.createElement('dt'); dt.textContent = k;
    const dd = document.createElement('dd'); dd.textContent = v;
    dl.appendChild(dt); dl.appendChild(dd);
  }
  tip.appendChild(dl);
  const nt = notesOf(p);
  if (nt) {                                   // 近傍語など，横に長い情報
    const n = document.createElement('p');
    n.className = 'notes';
    const b = document.createElement('b');
    b.textContent = nt[0] + ' ';
    n.appendChild(b);
    n.appendChild(document.createTextNode(nt[1]));
    tip.appendChild(n);
  }
}

// **原空間での近傍を線で結ぶ。** 画面の近さは射影の結果にすぎない。
// 線が遠くへ伸びるなら，その点の近傍関係は2次元に収まっていない。
// これを見せるのが，この図でいちばん大事なところである。
function drawLinks(p, w, h) {
  if (!links) return;
  while (links.firstChild) links.removeChild(links.firstChild);
  if (!p.links || !p.links.length) return;
  const NS = 'http://www.w3.org/2000/svg';
  for (const j of p.links) {
    const q = PTS[j];
    if (!q) continue;
    const ln = document.createElementNS(NS, 'line');
    ln.setAttribute('x1', p.x * w); ln.setAttribute('y1', p.y * h);
    ln.setAttribute('x2', q.x * w); ln.setAttribute('y2', q.y * h);
    links.appendChild(ln);
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', q.x * w); c.setAttribute('cy', q.y * h);
    c.setAttribute('r', 5);
    links.appendChild(c);
  }
}

let cur = null;
function show(p, px, py) {
  const w = box.clientWidth, h = box.clientHeight;
  if (p !== cur) { fill(p); drawLinks(p, w, h); cur = p; }
  ring.style.left = (p.x * w) + 'px';
  ring.style.top = (p.y * h) + 'px';
  ring.style.opacity = 1;
  tip.style.opacity = 1;
  // はみ出さないように寄せる
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lx = px + 16, ly = py + 14;
  if (lx + tw > w) lx = px - tw - 16;
  if (ly + th > h) ly = py - th - 14;
  tip.style.left = Math.max(0, lx) + 'px';
  tip.style.top = Math.max(0, ly) + 'px';
  rows.forEach(r => r.classList.toggle('on', r.dataset.i === String(p.r)));
}
function hide() {
  tip.style.opacity = 0; ring.style.opacity = 0; cur = null;
  if (links) while (links.firstChild) links.removeChild(links.firstChild);
  rows.forEach(r => r.classList.remove('on'));
}

box.addEventListener('pointermove', e => {
  const r = box.getBoundingClientRect();
  const p = nearest(e.clientX - r.left, e.clientY - r.top, r.width, r.height);
  if (p) show(p, e.clientX - r.left, e.clientY - r.top); else hide();
});
box.addEventListener('pointerleave', hide);

// 表の行にカーソルを乗せても，図の上の点が光る（逆引き）。
// **カーソルが使えない人にも同じ情報が届くように**，表を必ず添える
// （点が数千を超える図だけは表を絞る。絞ったことは図の下に明記する）。
rows.forEach(r => {
  r.addEventListener('mouseenter', () => {
    // 表の行は論理点。図の上では**先頭の面**の点を光らせる
    const p = PTS[Number(r.dataset.i)];
    if (!p) return;
    const w = box.clientWidth, h = box.clientHeight;
    show(p, p.x * w, p.y * h);
  });
  r.addEventListener('mouseleave', hide);
});

// 絞り込み。語・作品・時代のどれでも当たる
const q = document.getElementById('q');
if (q) q.addEventListener('input', () => {
  const s = q.value.trim();
  let n = 0;
  rows.forEach(r => {
    const hit = !s || r.textContent.includes(s);
    r.style.display = hit ? '' : 'none';
    if (hit) n++;
  });
  // 表を絞った図では，**表に無い語も図の上では当たる**。
  // 表の件数だけを出すと「無い」と誤解されるので両方を出す。
  const nlog = Number(document.body.dataset.nlog || rows.length);
  let extra = '';
  if (s && rows.length < nlog) {
    const seen = new Set();
    for (const p of PTS) if (p.term.includes(s)) seen.add(p.r);
    extra = '（図の上 ' + seen.size + ' 件）';
  }
  document.getElementById('count').textContent = n + ' 件' + extra;
  // **図の上にもマーカーを付ける。** 表だけ絞っても「どこにあるか」は分からない。
  if (!marks) return;
  while (marks.firstChild) marks.removeChild(marks.firstChild);
  if (!s) return;
  const NS = 'http://www.w3.org/2000/svg';
  const w = box.clientWidth, h = box.clientHeight;
  let drawn = 0;
  for (const p of PTS) {
    if (!p.term.includes(s)) continue;
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', p.x * w); c.setAttribute('cy', p.y * h);
    c.setAttribute('r', 7);
    marks.appendChild(c);
    if (++drawn > 400) break;          // マーカーが多すぎると図が読めない
  }
});
"""


def save_interactive(fig, ax, stem, xs, ys, tips, out=None, title='',
                     note='', table_cols=None, source=None, id_col='語',
                     hint='', table_idx=None, coords=None):
    """SVG を保存し，**同じ SVG を埋め込んだ対話的な HTML** も書く。

    ``xs`` ``ys`` はデータ座標，``tips`` は点ごとの情報
    （``{'term': 語, 'fields': [(見出し, 値), …]}`` の並び）。
    3つの長さは一致していなければならない。ずれたまま描くと，
    **指した点と出る語が食い違う**（注記の添字ずれと同じ事故）。

    位置は「図全体に対する割合」で書き出す。SVG を ``width:100%`` で
    伸縮させても割合は変わらないので，どんな幅でも点とポインタの判定範囲が
    合う。座標は matplotlib の変換を通して得るので，**図と HTML で
    座標の計算が二重にならない**。

    ``source`` に入力ファイルのパスを渡すこと。**どの表から描いた図かを
    HTML の末尾に刻む。** これが無いと，試験用の作りかけのデータから
    描いた図と，本番のデータから描いた図が見分けられない。
    入力がプロジェクトの外（``/tmp`` など）にあるときは
    「試験用」と赤字で出し，配布してはいけないことを図自身に言わせる。

    ``id_col`` は表の第1列の見出し（既定「語」。作品を点にする図では
    「作品」などに変える）。

    ``ax`` には**面の並び**も渡せる（``[axes[0], axes[1]]``）。同じ点を
    別の色分けで2面に描いた図では，どちらの面を指しても同じ情報が出る。
    表の行は点ごとに1行だけ作る（面の数だけ重複させない）。

    ``table_idx`` は**表に載せる点の添字**（既定は全点）。点が数千を超える
    図では表を全件出すと HTML が数 MB になり，読む側にも役に立たない。
    そのときは載せる点を選ぶ。**ただし図のポインタの判定範囲と検索は全点に効く**
    ので，表に無い語も指せるし検索で図にマーカーが付く。表を絞ったときは，
    何件のうち何件を載せたかを HTML に明記する（黙って捨てないこと）。

    ``coords`` は**面ごとの座標**（``[(x1, y1), (x2, y2)]``）。同じ点を
    **違う座標系**で2面に描いた図（主成分分析と UMAP の比較など）で使う。
    渡さなければ全部の面で ``xs`` ``ys`` を使う。
    ⚠ 面ごとに座標が違うのに ``coords`` を渡さないと，2面めのポインタの判定範囲が
    1面めの座標で置かれる。**図は出るが，指した点と出る語が食い違う。**
    """
    import json as _json
    if not (len(xs) == len(ys) == len(tips)):
        raise ValueError(
            f'save_interactive: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'情報={len(tips)}）。座標と情報を同じ添字で絞り込むこと。')

    svg_path = save_fig(fig, stem, out=out)
    outdir = svg_path.parent

    # ---- 埋め込む SVG は**切り取らずに**保存する ------------------------
    # save_fig は bbox_inches='tight' で余白を詰めるため，図全体に対する
    # 割合と，ファイルの座標系がずれる。埋め込み用は詰めずに出す。
    import io
    buf = io.StringIO()
    fig.savefig(buf, format='svg', dpi=RASTER_DPI)
    svg = buf.getvalue()
    svg = svg[svg.index('<svg'):]          # XML 宣言と DOCTYPE を除く

    # ---- 点の位置を図全体に対する割合で得る -----------------------------
    axes_list = list(ax) if isinstance(ax, (list, tuple, np.ndarray)) else [ax]
    W, H = fig.bbox.width, fig.bbox.height
    n_pts = len(tips)

    # **見出しは点ごとに書かない。** 点が1万個ある図では，
    # 「品詞」「頻度」…という見出しを1万回繰り返すだけで HTML が
    # 1 MB 以上ふくらむ。全点で見出しが同じなら1回だけ書き，
    # 値の並びだけを点に持たせる（JS 側で組み直す）。
    keys = [str(k) for k, _ in tips[0].get('fields', [])] if tips else []
    same_keys = bool(keys) and all(
        [str(k) for k, _ in t.get('fields', [])] == keys for t in tips)
    nheads = {str(t['notes'][0]) for t in tips if t.get('notes')}
    nhead = next(iter(nheads)) if len(nheads) == 1 else ''
    # 本文を渡さず ``notes=(見出し, None)`` としたときは，
    # ``links`` の先の語を JS 側で並べる
    linknotes = bool(nhead) and any(
        t.get('notes') and t['notes'][1] is None and t.get('links')
        for t in tips)

    pts = []
    if coords is not None and len(coords) != len(axes_list):
        raise ValueError(
            f'save_interactive: coords の数が面の数と違う'
            f'（面 {len(axes_list)} / coords {len(coords)}）')
    for k, axk in enumerate(axes_list):
      xk, yk = (coords[k] if coords is not None else (xs, ys))
      if not (len(xk) == len(yk) == n_pts):
          raise ValueError(
              f'save_interactive: 面 {k} の座標の数が情報の数と違う'
              f'（x={len(xk)}, y={len(yk)}, 情報={n_pts}）')
      pxy = axk.transData.transform(np.column_stack([np.asarray(xk, float),
                                                     np.asarray(yk, float)]))
      for j, (t, (px, py)) in enumerate(zip(tips, pxy)):
        i = k * n_pts + j
        # **変数名に注意。** ここを d と書くと，上で取った出力先 d
        # （svg_path.parent）を上書きして，最後に d / '....html' が
        # 「dict ÷ str」になる。名前は使い回さない。
        # 添字 i は JS では使わない（行は r で引く）。点が1万個ある図では
        # 使わない値も 100 KB 単位で効くので書かない。
        rec = {'r': j, 'term': str(t.get('term', '')),
               'x': round(float(px) / W, 6),
               'y': round(1 - float(py) / H, 6)}        # SVG は上が 0
        if same_keys:
            rec['v'] = [str(b) for _, b in t.get('fields', [])]
        else:
            rec['fields'] = [[str(a), str(b)] for a, b in t.get('fields', [])]
        if t.get('notes'):
            # ('見出し', '本文') の2つ組。横に長い情報（近傍語など）。
            # 本文を None にすると，**線で結ぶ先の語を JS が並べる**
            # （同じ語の列を点ごとに書かずに済む。1万点で 1 MB 近く効く）
            if t['notes'][1] is None:
                pass
            elif nhead:
                rec['n'] = str(t['notes'][1])
            else:
                rec['notes'] = [str(t['notes'][0]), str(t['notes'][1])]
        if t.get('links'):
            # 原空間での近傍の添字。**同じ面の中で**線を結ぶ
            rec['links'] = [k * n_pts + int(q) for q in t['links']]
        pts.append(rec)

    cols = table_cols or keys
    head = f'<tr><th>{_esc(id_col)}</th>' + ''.join(
        f'<th>{_esc(c)}</th>' for c in cols) + '</tr>'
    # 数字の列だけ右寄せにする。時代名や作品 ID を右寄せにすると読みにくい。
    def _numish(v):
        t = str(v).strip().replace('%', '').replace(',', '')
        t = t.lstrip('+-')
        return bool(t) and t.replace('.', '', 1).isdigit()

    # 表に載せる点。**面の数だけ重複させない**（論理点1つに1行）
    if table_idx is None:
        order = list(range(n_pts))
    else:
        seen, order = set(), []
        for i in [int(q) for q in table_idx]:   # 重複を除きつつ順序は保つ
            if 0 <= i < n_pts and i not in seen:
                seen.add(i); order.append(i)

    # 表は **tips から作る**（点の JSON は見出しを省いてあるので）
    body = []
    for j in order:
        fv = {str(a): str(b) for a, b in tips[j].get('fields', [])}
        tds = ''
        for c in cols:
            v = fv.get(c, '')
            cls = ' class="num"' if _numish(v) else ''
            tds += f'<td{cls}>{_esc(v)}</td>'
        body.append(f'<tr data-i="{j}">'
                    f'<td>{_esc(tips[j].get("term", ""))}</td>{tds}</tr>')

    # ---- 由来を図自身に刻む -------------------------------------------
    # **どの表から描いた図かが分からないと，試験用のデータで描いた図が
    # 本物として配られかねない。**
    import datetime as _dt
    stamp = _dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M')
    src = Path(source) if source else None
    prov = f'点 {len(pts)} 個／作図 {stamp}'
    warn = ''
    if src is not None:
        try:
            mt = _dt.datetime.fromtimestamp(src.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        except OSError:
            mt = '不明'
        prov = f'入力 {src.name}（更新 {mt}）／' + prov
        # プロジェクトの外（/tmp など）から描いた図は試験用である
        try:
            outside = not str(src.resolve()).startswith(str(ROOT.resolve()))
        except Exception:                               # noqa: BLE001
            outside = True
        if outside or '/tmp/' in str(src):
            warn = ('<p class="danger">⚠ <b>試験用の入力から作った図である。'
                    f'配布してはいけない。</b>（入力 {_esc(str(src))}）</p>')
            prov = f'入力 {_esc(str(src))}／' + f'点 {len(pts)} 個／作図 {stamp}'

    html = f"""<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{_esc(title or stem)}</title>
<style>{INTERACTIVE_CSS}</style></head>
<body data-nlog="{n_pts}" data-ntable="{len(body)}"><div class="wrap">
<h1>{_esc(title or stem)}</h1>
<p class="sub">{_rich(note)}</p>
{warn}
<div class="card">
  <div class="figbox">
    {svg}
    <svg id="links"></svg><svg id="marks"></svg>
    <div id="hit"></div><div id="ring"></div><div id="tip"></div>
  </div>
  <p class="hint">{_rich(hint or '点にカーソルを近づけると語が出る（最も近い点を拾うので，真上に置かなくてよい）。図の中の注記は静止版と同じものである。')}</p>
  <div class="bar">
    <input id="q" type="search" placeholder="語・作品・時代で絞り込む">
    <span id="count">{len(body)} 件</span>
    <span>·</span>
    {(f'<span>表は {len(body)} 件（図の点は {n_pts} 件。'
      '表に無い語も図の上で指せる。検索は図のマーカーにも効く）</span><span>·</span>')
     if len(body) < n_pts else ''}
    <a href="{_esc(svg_path.name)}" download>SVG を保存</a>
    <span>（この HTML の中の図はその SVG そのもの）</span>
  </div>
  <div class="scroll"><table><thead>{head}</thead>
    <tbody>{''.join(body)}</tbody></table></div>
  <p class="prov">{prov}</p>
</div>
<script type="application/json" id="pts-data">{_json.dumps(
    {'keys': keys if same_keys else [], 'nhead': nhead,
     'linknotes': linknotes, 'pts': pts},
    ensure_ascii=False, separators=(',', ':'))}</script>
<script>{INTERACTIVE_JS}</script>
</div></body></html>
"""
    path = outdir / f'{stem}.html'
    path.write_text(html, encoding='utf-8')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB・対話版／'
          f'点 {len(pts)} 個)')
    return svg_path, path


def _esc(s):
    """HTML の特殊文字をエスケープする。**語はコーパス由来なので必ず通す。**"""
    return (str(s).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


def _rich(s):
    """注記の ``**…**`` だけを太字にする。

    説明文をノートブックと同じ書き方（Markdown 風）で書けるようにする。
    **先に必ず _esc を通す**ので，タグを書き込まれる余地は無い。
    ``**`` のままだと HTML では記号がそのまま出て読みにくい。
    """
    import re as _re
    return _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', _esc(s))


PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
           '#56B4E9', '#D55E00', '#F0E442', '#666666']

# --- 順序のあるものを色分けするための1色相のランプ -----------------------------
# **時代・年次・段階のように順序のあるものを，上の8色で色分けしてはいけない。**
# 明治中期が青で明治後期が黄なら，隣り合う時代が隣り合う色にならず，
# 「時代が下るとどちらへ動くか」という肝心のことが読めなくなる。
# 1色相の濃淡にすれば，近いもの同士が近い色になり，勾配がそのまま見える。
# 散布図の点は白地の上に置くので，いちばん明るい段は 100 ではなく
# 250（背景との対比 2:1）から始める。100 は面を色分けするとき（ヒートマップ）用。
SEQ_BLUE_STEPS = ['#86b6ef', '#5598e7', '#3987e5',
                  '#256abf', '#184f95', '#0d366b']
SEQ_BLUE = LinearSegmentedColormap.from_list('jlit_blue', SEQ_BLUE_STEPS)

# 離散の順序（4区分など）を色分けするときはこちら。隣の段と明度差が十分あり，
# いちばん明るい段も背景から浮く（対比 2:1 以上）ことを確かめてある。
SEQ_BLUE_5 = ['#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b']

# 大分類の2色。散布図ではどの2点も隣り合いうるので**全ペアが
# 見分けられる必要**があり，使える色数は多くない。2色に絞って，
# 下位の区別はマーカーの形に持たせる。
GENRE_C = {'Fiction': '#2a78d6', 'Nonfiction': '#eb6834'}

# --- 初出年の5段 ---------------------------------------------------------
# 切れ目は period と同じ 1900／1912／1926／1945。
# **6段にはできない。** 1色相の濃淡で順序を見せるには，隣り合う段の明度差が
# 0.06 以上要る。この青系ランプは 250→700 で明度差にして 0.30 ほどしか幅が
# 無いので，段を6つ取るとどこかが 0.06 を下回り（0.05 台），隣が見分けられなくなる。
# そこで作品数3点の明治前期（〜1886）を明治中期にまとめて5段とする。
YEAR_EDGES = [1900, 1912, 1926, 1945]
YEAR_LABELS = ['〜1899 明治前・中期', '1900-1911 明治後期', '1912-1925 大正',
               '1926-1944 昭和戦前', '1945- 昭和戦後']


def year_bands(years):
    """初出年を5段にまとめ，``(段番号, ラベル, 色)`` を返す。

    段番号は 0〜4。**初出年が読めないものは -1** にする。0 に割り当てると
    年の分からない作品が全部いちばん古い段に入り，通時の議論が崩れる。

    時代で色分けする図はすべてこれを通すこと。同じ色が全ステップで同じ時代を
    指すようになり，Step 1 の図と Step 7 の図を並べて読める。
    """
    y = pd.to_numeric(pd.Series(list(years)), errors='coerce')
    code = np.full(len(y), -1, dtype=int)
    ok = y.notna().values
    if ok.any():
        code[ok] = np.digitize(y[ok].values, YEAR_EDGES)
    return code, YEAR_LABELS, SEQ_BLUE_5


def run_script(script, *args, tail=4000):
    """scripts/ のスクリプトを実行し，標準出力・標準エラー・終了コードを必ず表示する。

    print(r.stdout or r.stderr) では，標準出力が空でないときに
    エラーの内容が隠れてしまう。学習用には両方見えるほうがよい。
    """
    cmd = [sys.executable, str(ROOT / 'scripts' / script)] + [str(a) for a in args]
    print('$ python', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, capture_output=True, text=True)

    def _tail(s):
        # 末尾 tail 文字だけを出す。行の途中で切らないよう，切ったときは
        # 次の改行から始め，前を省いたことを明示する
        if len(s) <= tail:
            return s
        s = s[-tail:]
        return '（…前略）\n' + s[s.find('\n') + 1:]

    if r.stdout:
        print(_tail(r.stdout))
    if r.stderr.strip():
        # 標準エラーには**エラー以外**も出る。MALLET は学習の進み具合
        # （<10> LL/token: …）と途中のトピック上位語をここに書く。
        # 判断は [exit 0] かどうかで行う
        print('--- stderr（進行ログを含む。エラーとは限らない）---')
        print(_tail(r.stderr))
    print(f'[exit {r.returncode}]' + ('' if r.returncode == 0 else '  ← 0 でなければ失敗'))
    return r


# 使うメタデータ。増補分を含む v3 があればそちらを優先する。
# v2 は v1 の 64 点しか無いので，増補後のコーパスで v2 を使うと
# 突合が外れて period も genre も空になる（Step 3 で v3 を作る）。
# Step 3 で自分が作った v3 は *_local.csv に書かれる（配布版は上書きしない。
# 上書きすると git pull のたびに衝突する）。自分の版 > 配布版 v3 > v2 の順。
for _m in ('corpus_metadata_v3_local.csv', 'corpus_metadata_v3.csv',
           'corpus_metadata_v2.csv'):
    META = ROOT / 'metadata' / _m
    if META.exists():
        break

# 図・表の書き出し先は自分の作業フォルダ my_work/results/。my_work/ は
# コースのリポジトリの外扱い（.gitignore）で，自分の GitHub にバックアップを取る。
OUT = ROOT / 'my_work' / 'results'
OUT.mkdir(parents=True, exist_ok=True)
print('OUT  =', OUT)


In [ ]:
from lib.aozora import normalise_iteration_marks, expand_kana_odoriji, expand_kunoji

samples = ['たゞ一人','つゞき','ホホホヽヽ','ドヾドン','はゝゝゝ','あゝ',
           'いろ／＼','とき／″＼','しみ〴〵','人々','時々','我々']
for s in samples:
    out, st = normalise_iteration_marks(s)
    print(f'{s:<10} → {out:<10}  {st}')

### 規則

| 記号 | 規則 | 例 |
|---|---|---|
| `ゝ` | 直前の平仮名を反復（直前が濁音なら清音化） | `はゝ` → `はは` |
| `ゞ` | 直前の平仮名を濁音化して反復 | `たゞ` → `ただ` |
| `ヽ` `ヾ` | 片仮名版 | `ドヾ` → `ドド` |
| `／＼` `〳〵` | 直前の仮名連続（2–4字）を反復 | `いろ／＼` → `いろいろ` |
| `／″＼` `〴〵` | 同上・濁音化 | `しみ〴〵` → `しみじみ` |
| `々` | **展開しない** | `人々` → `人々` |

くの字点は「直前の何字を繰り返すか」が原理的に曖昧なので，
**機械推定であることを記録に残す**（`@resp="auto"`）。

## 1. 正規化の実行

`04_normalise.py` は XML から4つのストリームを作る。

| ストリーム | 内容 | 使いどころ |
|---|---|---|
| `full` | 地の文＋会話 | 既定の分析対象 |
| `narration` | 地の文のみ | 語りの人称を測るとき |
| `speech` | 会話のみ | 会話文比率・役割語の分析 |
| `embedded` | 書簡・手記など埋め込みテクスト | 枠構造をもつ作品の内部対照 |

いずれも `<note>` は除外，ルビの読みは除外，`<g>` は復元文字に置換される。

`embedded` は `<quote type="embedded">`，すなわち8段落以上にわたる引用である。
『こころ』下「先生と遺書」（本文の 51%），森鴎外『即興詩人』（30%），
夢野久作『ドグラ・マグラ』（6%）などが該当する。これらを地の文と
一緒くたにすると，枠物語の内と外の文体差が見えなくなる。

リポートの `speech_ratio` は，会話標示が `full` でない作品では**空欄**になる。
該当が何点あるかは `conversion_report.csv` の `speech_markup` 列を
数えること（増補で変わる）。
`fillna(0)` をしないこと。

In [ ]:
XML   = ROOT/'data'/'xml'
PLAIN = ROOT/'data'/'plain'
if need(XML, 'Step 2 のノートブックを最後まで実行し，data/xml/ に XML を作ること'):
    run_script('04_normalise.py', '--in', XML, '--out', PLAIN,
               '--config', ROOT/'config'/'pipeline.yaml')
else:
    print('XML がありません。Step 2 のノートブックを先に実行してください。')

In [ ]:
p = PLAIN/'normalise_report.csv'
if need(p, '04_normalise.py のセルを先に実行すること'):
    nr = pd.read_csv(p)
    print('踊り字の展開合計:',
          int(nr.kana_odoriji_expanded.sum()), '(仮名)',
          int(nr.kunoji_expanded.sum()), '(くの字)')
    print('未解決外字:', int(nr.unresolved_gaiji.sum()))
    LAB = work_labels()
    nr2 = nr.nlargest(10,'kana_odoriji_expanded').copy()
    nr2.insert(1, '作品',
               nr2['file'].str.replace(r'\.(xml|txt)$','',regex=True).map(LAB))
    show(nr2[['作品','kana_odoriji_expanded','kunoji_expanded',
              'unresolved_gaiji','speech_ratio']]
         .rename(columns={'kana_odoriji_expanded':'仮名踊り字',
                          'kunoji_expanded':'くの字点',
                          'unresolved_gaiji':'未解決外字',
                          'speech_ratio':'会話文比率'}),
         caption='踊り字の展開が多かった作品（上位10件）',
         fmt={'会話文比率':'{:.1%}'})
    print('**偏りが極端なら，文体差ではなく底本の正書法差である。**'
          '同じ作家の他の作品と比べること。')

### 演習 — 枠の内と外で文体は違うか

『こころ』は上・中が「私」の語り，下が先生の遺書である。遺書は
`<quote type="embedded">` として切り出されているので，同じ作品の**内部で**
文体を対照できる。作家も時代も同じなのだから，差が出ればそれは
**枠構造そのものの効果**である。

`narration`（地の文）と `embedded`（埋め込みテクスト）を比べてみよう。
ここでは指標として，一人称代名詞の頻度と平均文長を使う。

In [ ]:
import re as _re

def profile(path):
    if not path.exists():
        return None
    t = path.read_text(encoding='utf-8')
    sents = [s for s in _re.split(r'[。！？]', t) if s.strip()]
    return {'chars': len(t),
            '私/万字': t.count('私') / max(1, len(t)) * 10000,
            '自分/万字': t.count('自分') / max(1, len(t)) * 10000,
            '平均文長': (sum(len(s) for s in sents) / len(sents)) if sents else 0}

ROWS = work_rows()          # 000148_000773 → メタデータの行（全件から）
LAB  = work_labels()        # 000148_000773 → 夏目漱石『こころ』

rows, miss = [], []
for f in sorted((PLAIN/'embedded').glob('*.txt')) if (PLAIN/'embedded').exists() else []:
    emb = profile(f)
    if not emb or emb['chars'] < 3000:        # 短すぎる切片は比較に耐えない
        continue
    nar = profile(PLAIN/'narration'/f.name)
    if not nar:
        continue
    stem = f.stem
    r = ROWS.get(stem)
    if r is None:
        miss.append(stem)
    # work_rows() が返すのは pandas の Series である。
    # **`r or {}` と書いてはいけない。** Series は真偽値に変換できず
    # ValueError になる（空でないかどうかが一意に決まらないため）。
    def col(name, default=''):
        return default if r is None else r.get(name, default)
    # ファイル名だけでは「誰の何の枠物語か」が分からず，
    # 「差は枠構造の効果か，埋め込みテクストのジャンルの効果か」という
    # 問いに答えようがない。作者・作品名・初出年を出す。
    rows.append({'作者': col('author_ja', '—'),
                 '作品': col('title_aozora', stem),
                 '初出': col('year_first'),
                 'embedded字数': emb['chars'], 'narration字数': nar['chars'],
                 '埋込比': round(emb['chars'] / max(1, emb['chars'] + nar['chars']), 3),
                 '私(emb)': round(emb['私/万字'],1), '私(nar)': round(nar['私/万字'],1),
                 '文長(emb)': round(emb['平均文長'],1),
                 '文長(nar)': round(nar['平均文長'],1),
                 'work_stem': stem})
if miss:
    print(f'[warn] メタデータに無い作品 {len(miss)} 件（語幹のまま表示）: '
          + '，'.join(miss[:5]))
if rows:
    show(pd.DataFrame(rows).sort_values('embedded字数', ascending=False)
           .drop(columns=['work_stem']),
         caption='枠の内（embedded）と外（narration）の比較',
         fmt={'embedded字数':'{:,.0f}','narration字数':'{:,.0f}',
              '埋込比':'{:.1%}','初出':'{:.0f}'})
    print('問い: 差が出た作品はどれか。その差は枠構造の効果と言えるか，')
    print('      それとも埋め込みテクストの**ジャンル**（書簡・手記）の効果か。')
else:
    print('embedded ストリームがありません。config/pipeline.yaml の streams を確認。')

## 2. UniDic の語形 — どれを分析単位にするか

これは**結果を左右する決定**である。

| 素性 | 「噓を」 | 「渋江」 | 「云つた」 |
|---|---|---|---|
| surface（表層形） | 噓 | 渋江 | 云つ |
| orthBase（書字形基本形） | 噓 | 渋江 | 云う |
| **lemma（語彙素）** | 嘘 | **シブエ** | 言う |
| lForm（語彙素読み） | ウソ | シブエ | イウ |

`lemma` は異表記（噓／嘘，云う／言う）を統合するので**通時比較に向く**。
ただし**固有名詞では片仮名の読みになる**という UniDic の仕様がある。

本授業の既定（`--lemma-policy mixed`）:

```
固有名詞     → orthBase（なければ表層形）
それ以外     → lemma（なければ orthBase → 表層形）
```

加えて `私-代名詞` のような同形異義接尾辞は既定で除外する。

In [ ]:
import fugashi
# 辞書は自分で探さず，05_tokenise_unidic.py と**同じ規則**で解決する。
# ここで別の辞書を読み込むと，このノートブックの表と data/tokens の中身が
# 違う辞書のものになる。（ファイル名が数字で始まるので import できない）
spec = importlib.util.spec_from_file_location(
    'tok', ROOT/'scripts'/'05_tokenise_unidic.py')
tok = importlib.util.module_from_spec(spec); spec.loader.exec_module(tok)

DIC, SRC = tok.resolve_dicdir(None)
NAME, VER = tok.identify_dict(DIC) if DIC else ('（未検出）', '')
tagger = fugashi.Tagger(f'-d {DIC}' if DIC else '')
print(f'辞書: {NAME} {VER}   ← {SRC}\n      {DIC}')
if NAME != tok.PROD_DICT:
    print(f'\n[warn] **本番の辞書ではない。** 本番は {tok.PROD_DICT} '
          f'{tok.PROD_VERSION}（docs/dictionary_comparison.md §10）。\n'
          f'       この辞書で出した数値は他の人の結果と比べられない。\n'
          f'       docs/00_setup_students.md §1.4 を見て入れ直すこと。')

s = 'ただ一人、しみじみと噓をついた。渋江抽斎の述志の詩である。'
rows = [{'surface':w.surface, 'pos1':w.feature.pos1, 'pos2':w.feature.pos2,
         'lemma':w.feature.lemma, 'orthBase':w.feature.orthBase,
         'lForm':w.feature.lForm} for w in tagger(s)]
show(pd.DataFrame(rows), caption=f'「{s}」の解析結果（{NAME} {VER}）')

### 演習 1 — 方針を変えると結果がどう変わるか

同じテクストを4つの `lemma-policy` で解析し，異なり語数を比べてみよう。
**どれが最も語をまとめるか。まとめすぎて困るのはどんな場合か。**

In [ ]:
# tok は上のセルで読み込んである（05_tokenise_unidic.py）
demo = ('その男は云つた。「私は噓をつかない」と。渋江抽斎は述志の詩を作つた。'
        'ただしみじみと思ふ。人々は時々さう言ふ。')
for pol in ['surface','orthBase','lemma','mixed']:
    keys = [tok.lemma_key(w, pol) for w in tagger(demo)]
    print(f'{pol:<10} 異なり{len(set(keys)):>3}  {" ".join(keys[:22])}')

## 3. どの辞書で解析するか — これは研究設計の決定である

**解析器（MeCab）と辞書（UniDic）は別物**で，辞書を替えれば結果は変わる。
国語研は変種ごとに別の辞書を配っている。本コーパスは 1872–1959 年の
口語小説が大半なので，**現代語の辞書が最善とはかぎらない**。

そこで 2026-09-22 に 4 辞書を 111 点で測り比べた。

| 辞書 | 未知語率 | 平均語長 | 助動詞率 |
|---|---:|---:|---:|
| **`unidic-novel`（近現代口語小説）← 本番** | **0.17%** | 1.575 | 10.9% |
| `unidic-qkana`（旧仮名口語） | 0.19% | 1.577 | 10.9% |
| `unidic-kindai-bungo`（近代文語）← 検算用 | 0.25% | 1.582 | **11.2%** |
| `unidic-cwj`（現代書き言葉） | 0.66% | 1.560 | 10.6% |

### ⚠ ここが勘所 — 未知語率だけを見てはいけない

辞書は**細かく刻めば未知語率を下げられる**。「知らない語」を
「知っている短い語2つ」に割れば，未知語は消えて見える。
だから未知語率と**平均語長**を必ず並べて見る。

上の表では，未知語率が最小の `novel` が cwj **より語が長い**
（1.575 > 1.560）。細かく分割して稼いだのではない，と言える。この確認をせずに
「未知語率が低いから良い辞書だ」と書くのは誤りである。

⚠ **判断の規則は測る前に決めておく。** 測ってから決めると，出た数字に
都合の良い規則を選ぶことになる。→ `docs/dictionary_comparison.md` §4

> **辞書は作品ごとに替えてはいけない。** 文語作品だけ近代文語 UniDic で
> 解析すれば，その作品の未知語率は下がる。しかし語彙素の体系が変わるので，
> **他の作品と混ぜて頻度比較できなくなる**。辞書はコーパス全体で統一し，
> 近代文語 UniDic は A_文語体の助動詞率の**検算**にのみ使う。

## 4. 形態素解析の実行

In [ ]:
TOK = ROOT/'data'/'tokens'
if need(PLAIN/'full', 'このステップの 04_normalise.py のセルを先に実行すること'):
    run_script('05_tokenise_unidic.py', '--in', PLAIN/'full', '--out', TOK,
               '--dicdir', DIC, '--lemma-policy', 'mixed')

### 演習 2 — 未知語率で前処理を評価する

未知語率は**前処理の良し悪しを測る最良の単一指標**である。
高いファイルには理由がある。旧仮名か，外字が残っているか，漢文脈か。

In [ ]:
p = TOK/'tokenise_report.csv'
if need(p, '05_tokenise_unidic.py のセルを先に実行すること'):
    tr = pd.read_csv(p)
    meta = load_meta()
    LAB = work_labels()
    print('未知語率 中央値 = {:.2%}'.format(tr.unknown_rate.median()))
    top_unk = tr.nlargest(12,'unknown_rate')[['file','tokens','unknown_rate']].copy()
    # ファイル名のままでは「なぜ未知語が多いのか」を考えられない
    top_unk.insert(1, '作品',
                   top_unk['file'].str.replace(r'\.txt$','',regex=True).map(LAB))
    show(top_unk.drop(columns=['file']).rename(columns={'tokens':'語数',
                                                        'unknown_rate':'未知語率'}),
         caption='未知語率の高い作品（上位12件）',
         fmt={'語数':'{:,.0f}','未知語率':'{:.2%}'})

    # 未知語は「何が何回」だけでなく**上位が全体の何割を占めるか**を見る。
    # 上位20語で大半を占めるなら，直すべき対象は少数に絞れる。
    unk = pd.read_csv(TOK/'unknown_words.csv')
    u = unk.head(20).copy()
    u.insert(0, '順位', range(1, len(u)+1))
    u['累積割合'] = u.freq.cumsum() / unk.freq.sum()
    show(u.rename(columns={'surface':'語', 'freq':'頻度'}),
         caption=f'未知語 上位20（異なり {len(unk):,} 語・延べ {unk.freq.sum():,}）',
         fmt={'頻度':'{:,.0f}', '累積割合':'{:.1%}'})
    grid(unk.surface.head(60).astype(str), ncol=10,
         caption='未知語 上位60（種類を眺めるため）')

**読み方。** 未知語の上位に来るものは5種類に分かれ，
**どれに当たるかで対処が違う。**

| 種類 | 意味 | 対処 |
|---|---|---|
| **カタカナ外来語** | **正常**。外来語は開いたクラスで，辞書に全部は入らない | 何もしない |
| 人名・地名 | **正常**。固有名詞は辞書に限界がある | 何もしない |
| `ゝ` `ゞ` を含む列 | **正規化漏れ** | `04` の設定を見直す |
| `※` `〓` | **外字が復元できていない** | `03` の変換を見直す |
| 文語の活用形（`けれ` `ざり` `べかり`） | **辞書の被覆の問題** | 辞書の選定で解く（上の §3） |

### ⚠ 種類が変わると，未知語率の**意味**も変わる

前処理の失敗（踊り字・外字）が消えて**外来語だけが残った状態**では，
未知語率はもう「前処理の質」を測っていない。**外来語の密度**を
測っている。

そして外来語密度は**時代とジャンルの変数**である。昭和戦前の探偵小説・
SF（海野十三）は多く，明治の文語論説（福沢諭吉）はほぼ無い。

> **この転換点を見落とすと，海野十三の未知語率の高さを
> 「解析の失敗」と報告に書いてしまう。**

自分の未知語リストがどちらの段階にあるかは，目で見るのではなく数える。

```
python3 scripts/14_unknown_profile.py --unknown data/tokens/unknown_words.csv --report data/tokens/tokenise_report.csv
```

種類ごとの割合と，**カタカナ表記の揺れ**（`ラムプ`／`ランプ`，
`ヰスキー`／`ウイスキー`）の組も出る。揺れは同じ語を別の型として
数えるので異なり語数を膨らませるが，**まとめるかどうかは研究の問いに
よる。表記の揺れ自体が正書法の近代化の資料でもある。**

## 5. データセットの構築 — 長さの偏りを吸収する

v1 の作品長は 30,555 語（鴎外『大塩平八郎』）から 502,937 語（藤村『夜明け前』）まで
**16.5倍**の開きがある。作品を1文書としてトピックモデルにかけると，
長篇1作が数トピックを独占する。

対策は2段構え。

1. 一定語数（既定 2,000 語）のチャンクに分割
2. 1作品あたりのチャンク数に上限を設ける（`--max-chunks`）

### その前に — メタデータを増補分まで広げる

`metadata/corpus_metadata_v2.csv` は **v1 の 64 点**しか載っていない。
増補後のコーパスはそれより多いので，差分の作品には行が無い。行が無いと
次のチャンク分割でメタデータの突合が外れ，**`period` も `genre` も空の
チャンク索引**ができあがる。しかもエラーは出ない。そのまま進むと
時代別 keyness も doc2vec のカテゴリー効果もトピックの通時変化も，
**すべて空振りしたまま最後まで通ってしまう。**

`00_extend_metadata.py` が，青空文庫の索引（`fetch_log.csv`）・増補候補表・
**トークン列からの実測**を典拠ごとに分けて v3 を作る。書き出し先は
**`metadata/corpus_metadata_v3_local.csv`**（自分の版。git には入らない）で，
配布版の `corpus_metadata_v3.csv` は書き換えない（書き換えると `git pull` の
たびに衝突する）。以後のセルは自分の版を優先して読む。
実測に渡すのは `data/tokens/tokens_surface/` であって `data/plain/full/` ではない。
後者は分かち書きされていないので，`str.split()` が段落を語として数えてしまう。
**機械で決められない列（`narration`, `register_level`, `audience`, `form`）は
`TBD` のまま残る。** これは手抜きではなく，語りの視点や読者層は本文を読まないと
決まらないからである。`*_needs_review.csv` に一覧が出るので，
自分の分析に使う列だけでも埋めてから先へ進むこと。

In [ ]:
# --tokens には **分かち書きされた** トークン列を渡すこと。
# data/plain/full を渡すと，空白で区切られていないので段落数を語数として
# 数えてしまい，ttr_x1000（TTR×1000）が 1000 近くになる（スクリプトが検査して止める）。
# --remeasure-all は既存64点も同じ方法で測り直す。v1 のテクストには
# 外字欠落と奥付混入があるので，本来はこちらが正しい。
run_script('00_extend_metadata.py',
           '--meta', ROOT/'metadata'/'corpus_metadata_v2.csv',
           '--fetch-log', ROOT/'data'/'aozora'/'fetch_log.csv',
           '--candidates', ROOT/'metadata'/'expansion_candidates.csv',
           '--tokens', TOK/'tokens_surface',
           '--remeasure-all',
           '--out', ROOT/'metadata'/'corpus_metadata_v3_local.csv')
# 自分の v3。配布版 corpus_metadata_v3.csv は書き換えない（git pull で衝突しない）
META = ROOT/'metadata'/'corpus_metadata_v3_local.csv'   # 以後はこちらを使う

### 増補で代表性はどれだけ改善したか — Step 1 の図を作り直す

Step 1 では，v1（64点）の偏りを表と2枚の図（時代の構成・文語 ⇄ 口語の散布図）で
確かめた。増補した v3 で同じ図を作り直し，**v1 と並べて**どこが埋まり，どこが
まだ埋まっていないかを目で確かめる。

比べたいのは**コーパスの構成**の違いだけなので，**両方とも v3 の実測値**を使う。
v1 側は v3 のうち v1 から引き継いだ64点（`file_v1` がある行），v3 側は分析に使う全行である。
Step 1 の図（v2 の値）と数値が少し違うのは，本番の辞書（unidic-novel）で
測り直したためで，物差しをそろえてから比べるためにそうしている。

読むときの問い：

1. どの時代・文体の**空白が埋まった**か。作品数と語数の両方で見る
2. **まだ偏っている**項目はどれか（下の表の「最大の割合」が 0.5 を超えるもの）
3. 文語 ⇄ 口語の図で，増補した作品（黒い縁取り）はどの時代のどのあたりに入ったか。
   各段の中央値の軌跡は変わったか

まだ偏っている項目は，Step 4 以降の分析で**言えることの限界**になる。
最終リポートの「データ」の節で触れること。

In [ ]:
# ---- v1 と v3 を同じ物差しで並べる ---------------------------------------
# 両方とも v3 の実測値を使う（構成の違いだけを見るため）。
#   v1 側 = v3 のうち v1 から引き継いだ64点（file_v1 がある行）
#   v3 側 = 分析に使う全行（load_meta が superseded・merged・too_short を除外する）
if need(META, 'すぐ上の 00_extend_metadata.py のセルを先に実行すること'):
    m3_all = load_meta(META, analysis_only=False)
    has_v1 = m3_all['file_v1'].fillna('').astype(str).str.strip() != ''
    m1 = m3_all[has_v1].reset_index(drop=True)
    m3 = load_meta(META)
    m3['added'] = m3['file_v1'].fillna('').astype(str).str.strip() == ''
    SIDES = [('v1（増補前）', m1), ('v3（増補後）', m3)]
    print(f'v1 側 {len(m1)} 点 ／ v3 側 {len(m3)} 点（うち増補 {int(m3.added.sum())} 点）')

    # 1) Step 1 と同じ「偏りの一覧」を並べる。TBD（未記入）は区分に数えない
    COLS = ['period','style_class','kana_orthography','ndc',
            'genre_main','audience','register_level','narration','author_sex']
    rows = []
    for col in COLS:
        r = {'項目': col}
        for name, d in SIDES:
            v = d[col].astype(str).replace({'nan': 'TBD'})
            vc = v[v != 'TBD'].value_counts()
            r[f'{name} 最大の区分'] = str(vc.index[0]) if len(vc) else ''
            r[f'{name} 最大の割合'] = float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan
            r[f'{name} 未記入'] = int((v == 'TBD').sum())
        rows.append(r)
    t = pd.DataFrame(rows)
    t['変化'] = t['v3（増補後） 最大の割合'] - t['v1（増補前） 最大の割合']
    show(t.sort_values('v3（増補後） 最大の割合', ascending=False),
         caption='偏りの一覧 — v1 と v3（最大の区分の割合が下がれば偏りが和らいだ）',
         fmt={'v1（増補前） 最大の割合': '{:.1%}', 'v3（増補後） 最大の割合': '{:.1%}',
              '変化': '{:+.1%}'})

    # 2) 時代の構成を作品数と語数の両方で
    per = []
    for name, d in SIDES:
        g = d.groupby('period').agg(作品数=('id', 'count'), 語数=('tokens', 'sum'))
        g['作品数の割合'] = g['作品数'] / g['作品数'].sum()
        g['語数の割合'] = g['語数'] / g['語数'].sum()
        g['版'] = name
        per.append(g.reset_index())
    per = pd.concat(per, ignore_index=True)
    show(per.rename(columns={'period': '時代'})[['時代', '版', '作品数', '作品数の割合', '語数', '語数の割合']]
            .sort_values(['時代', '版']),
         caption='時代の構成 — v1 と v3',
         fmt={'作品数の割合': '{:.1%}', '語数': '{:,.0f}', '語数の割合': '{:.1%}'})

In [ ]:
# ---- 図1：時代の構成（Step 1 の Step1_period_balance を v1・v3 で並べる）----
if 'per' in globals():
    periods = sorted(per['period'].unique())
    x = np.arange(len(periods))
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True)
    for ax, col, ttl in [(axes[0], '作品数の割合', '作品数の比率'),
                         (axes[1], '語数の割合', '語数の比率')]:
        for k, (name, _) in enumerate(SIDES):
            s = per[per['版'] == name].set_index('period')[col].reindex(periods).fillna(0)
            ax.bar(x + (k - 0.5) * 0.4, s.values, 0.38, label=name,
                   color=['#c3c2b7', PALETTE[0]][k])
        ax.set_xticks(x)
        ax.set_xticklabels([p.split('_', 1)[1] for p in periods], rotation=25, ha='right')
        ax.set_title(ttl)
        ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', alpha=.25)
    axes[0].set_ylabel('比率'); axes[1].legend(frameon=False)
    fig.suptitle('時代区分の構成：v1（増補前）と v3（増補後）', y=1.02)
    fig.tight_layout(); save_fig(fig, 'Step3_period_balance_v1_v3'); plt.show()

In [ ]:
# ---- 図2：文語 ⇄ 口語（Step 1 の Step1_bungo_kogo を v1・v3 で並べる）-----
# 色・軸・中央値の軌跡は Step 1 と同じ作り方。両面で軸の範囲をそろえる。
# v3 の面では，増補した作品を黒い縁取りで示す。
if 'SIDES' in globals():
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), sharex=True, sharey=True)
    band_n = {}                       # 段ごとの点数（凡例ではなく表で示す）
    for ax, (name, d) in zip(axes, SIDES):
        last = ax is axes[-1]         # 凡例は右の面にだけ置く（色の意味は両面共通）
        code, blabels, bcols = year_bands(d.year_first)
        added = d['added'].values if 'added' in d else np.zeros(len(d), dtype=bool)
        for i, lab in enumerate(blabels):
            m = (code == i)
            if not m.any():
                continue
            edge = np.where(added[m], '#1f1f1f', 'white')
            ax.scatter(d.bungo_per10k[m], d.kogo_per10k[m], s=48, alpha=.9,
                       color=bcols[i], edgecolor=edge, linewidth=.9,
                       label=lab if last else None, zorder=3)
        band_n[name] = {lab: int((code == i).sum()) for i, lab in enumerate(blabels)}
        if (code < 0).any():
            m = (code < 0)
            band_n[name]['初出年不明'] = int(m.sum())
            ax.scatter(d.bungo_per10k[m], d.kogo_per10k[m], s=42, color='#c3c2b7',
                       edgecolor='white', linewidth=.8,
                       label='初出年不明' if last else None, zorder=3)
        mx = [d.bungo_per10k[code == i].median() for i in range(len(blabels))]
        my = [d.kogo_per10k[code == i].median() for i in range(len(blabels))]
        ax.plot(mx, my, color='#55554f', linewidth=1.4, alpha=.85, zorder=4,
                label='各段の中央値（古→新）' if last else None)
        ax.scatter(mx, my, s=120, marker='D', c=bcols, edgecolor='#55554f',
                   linewidth=1.2, zorder=5)
        ax.set_xscale('symlog', linthresh=10)
        ax.set_title(f'{name}：{len(d)} 点')
        ax.set_xlabel('文語助動詞標識（/万語・対数目盛）')
        ax.grid(alpha=.25, linewidth=.6, zorder=0)
        ax.spines[['top', 'right']].set_visible(False)
    axes[0].set_ylabel('口語助動詞標識（/万語）')
    axes[1].scatter([], [], s=48, color='white', edgecolor='#1f1f1f', linewidth=.9,
                    label='増補した作品（黒い縁取り）')
    axes[1].legend(frameon=False, fontsize=8.5, loc='upper left',
                   bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
    fig.suptitle('文語 ⇄ 口語：v1（増補前）と v3（増補後）'
                 '（1点＝1作品／色＝初出年・濃いほど新しい）', y=1.01)
    fig.tight_layout(); reserve_right(fig, 0.84)
    save_fig(fig, 'Step3_bungo_kogo_v1_v3'); plt.show()
    bn = pd.DataFrame(band_n).fillna(0).astype(int)
    bn['増えた点数'] = bn.iloc[:, 1] - bn.iloc[:, 0]
    show(bn.reset_index().rename(columns={'index': '初出年の段'}),
         caption='初出年の段ごとの作品数 — v1 と v3')

In [ ]:
DS = ROOT/'data'/'datasets'
if need(TOK/'tokens_content', 'このステップの 05_tokenise_unidic.py のセルを先に実行すること'):
    # --max-chunks は長篇の支配を防ぐための上限。結合後の『夜明け前』は
    # 約107チャンクあり，上限を外すと1作品でトピックを数本占有する。
    # 上限を超えた作品からは**層化無作為で**抜き，--seed で再現する。
    run_script('06_build_datasets.py', '--tokens', TOK/'tokens_content',
               '--meta', META, '--out', DS,
               '--chunk', 2000, '--max-chunks', 40,
               '--sample', 'stratified', '--seed', 20260920)

### 抜き出しの再現性

上限を超えた作品からどのチャンクを残すかは，**結果に効く決定**である。
論文には最低でも `--chunk` `--max-chunks` `--seed` の3つを書くこと。

乱数シードは `f'{seed}:{作品の語幹}'` として**作品ごとに**作っている。
`random.seed()` を最初に一度だけ呼ぶ実装だと，各作品の標本は
「それまでに何回 random を呼んだか」に依存する。**作品を1点足しただけで，
後ろに並ぶ全作品の標本が入れ替わり，前回の結果と比べられなくなる。**
作品ごとにシードを作れば，コーパスの構成が変わっても同じ作品からは同じ
チャンクが選ばれる。

`--sample` には4つある。

| 値 | 何をするか | いつ使うか |
|---|---|---|
| `stratified` | 作品を40等分し，各区画から1つ無作為に抜く（既定） | 標準。冒頭・中盤・結末が必ず入る |
| `random` | 作品全体から無作為に抜く | 層化の必要がないとき |
| `spread` | 等間隔に抜く | 乱数を使いたくないとき |
| `head` | 先頭から | 使わない。冒頭に偏る |

既定を `stratified` にしたのは，作品の全体から漏れなく抜くためである。単純無作為で107から40を
引くと，たまたま連続した区間が抜け落ちる（この設定では最大8チャンク＝
約16,000語）。物語の位置と主題が相関する長篇では，その区間の話題が
標本から丸ごと漏れる。層化すれば区間の抜けは最大5チャンクに収まり，
冒頭・中盤・結末が必ず標本に入る。

**層化しても無作為性は失われない**。区画の中でどのチャンクを引くかは
乱数が決めており，`spread`（等間隔＝決定論的）とは別物である。

In [ ]:
p = DS/'chunks_index.csv'
if need(p, '先に 06_build_datasets.py のセルを実行すること'):
    ci = pd.read_csv(p)
    print('チャンク総数:', len(ci))
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    for ax, col in zip(axes, ['period','genre_sub']):
        vc = ci[col].value_counts().sort_index() if col=='period' else \
             ci[col].value_counts().head(12)
        ax.barh([str(i)[:22] for i in vc.index][::-1], vc.values[::-1],
                color=PALETTE[0])
        ax.set_title(f'チャンク数: {col}'); ax.spines[['top','right']].set_visible(False)
    fig.tight_layout(); save_fig(fig, 'Step3_chunks'); plt.show()

## 5.5 KWIC コンコーダンサ — **テクストに戻る道具** ★★

数えたあとに本文へ戻れなければ，数えたことに意味は無い。**分析の最後の砦は
"back to texts" である。** 頻度表や word embedding は問いを作る道具で，答えは本文にある。

そこで，いま作った解析結果（`data/tokens/tsv`）から**索引**を作り，
ブラウザで使える KWIC コンコーダンサを起動する。

| できること | 中身 |
|---|---|
| **語彙素と表層形の切り替え** | 同じ解析結果の別の列。「語彙素で検索して表層形で読む」ができる |
| 品詞・活用形での指定 | `/動詞`・`言う/動詞`。`L:`／`S:` で項ごとに列を混ぜられる |
| 語の連なり | `汽車 に 乗る`。**文境界は越えない** |
| ワイルドカードと正規表現 | `乗*`・`*車`・`re:^汽.$` |
| **出典の表示** | 時代区分・著者・作品名（＋初出年・文体）。用例に出典が付く |
| 絞り込み | 時代区分・著者・文体・作品 |
| 並べ替え | 左1語・左1→2語・右1語・初出年・著者（本式のコンコーダンサと同じ） |
| 分布 | 作品別・時代区分別（**1万語あたりで正規化**） |
| 共起語 | 指標（LogDice・MI・MI3・t・G²・ΔP の両向き・共起頻度）を選んで上位を選び直す。**共起語の品詞**（大分類・中分類）で絞れる |
| **異綴形** | 語彙素で引くと，書字形基本形ごとの件数と，時代区分・作家・作品ごとの内訳が出る。コーパス全体で表記の揺れが大きい語彙素の一覧も出せる |
| **本文へ広げる** | 用例の行を押すと前後の文が開く。語にカーソルを当てると語彙素・品詞・活用形・語種が出る |
| 書き出し | CSV と，そのまま配れる HTML |
| **操作マニュアル** | 画面右上の「操作マニュアル ↗」で**別のウィンドウ**に開く。各欄の「？」でその欄の説明に飛ぶ。画面と左右に並べて使うとよい |

⚠ **索引は 05 を実行し直したら作り直す。** 辞書を替えれば切り方が変わるので，
古い索引のままでは用例が本文と食い違う。索引には辞書名と版が刻まれていて，
画面の下に常に出る。**どの辞書で切った本文を読んでいるか分からない用例は，
証拠にならない。**

In [ ]:
# ---- 索引を作る（05 を実行し直したら，これも作り直す）----------------
KW = ROOT/'data'/'kwic'
if need(TOK/'tsv', 'このステップの 05_tokenise_unidic.py のセルを先に実行すること'):
    run_script('15_kwic_index.py', '--tsv', TOK/'tsv', '--meta', META,
               '--out', KW, tail=2500)

### 画面を起動する

次のセルは**別のプロセス**としてサーバを起動する（ノートブックは止まらない）。
出てくるリンクをクリックすると，ブラウザで開く。

サーバはカーネルを再起動しても止まらない。使い終わったら下の「止める」の
セルで `STOP_KWIC = True` にして実行する（**既定では止めない**。上から順に
実行したとき，起動した直後に止めてしまわないようにしてある）。

⚠ サーバは **127.0.0.1 でだけ待ち受ける**（このマシンからしか見えない）。
認証の無い簡易サーバなので，学内のネットワークに公開しないための仕様である。127.0.0.1 は
マシンごとに別のものなので，DH ラボの iMac で隣の人と同じ `PORT`（8765）を
使っても衝突しない。ただし，前にそのマシンを使った人がログアウトせずに離れる
（ファストユーザースイッチ）と，その人のサーバが 8765 に残っていることがある。
下のセルは**自分の索引のサーバかどうかを確かめ**，違えば 8766, 8767, … の
空いているポートを自動で使う。`PORT` を手で変える必要はない。

In [ ]:
# ---- 画面を起動する ----------------------------------------------------
# **ポートが開くまで待ってからリンクを出す。** 待たずにリンクを出すと，
# 索引を読んでいる最中にクリックして「サーバに接続できません」になる。
# 起動しなかったときはログをその場に出す。
import socket, subprocess, sys, time, urllib.request
from IPython.display import display, HTML
import getpass, json, os
PORT0 = 8765         # 既定のポート番号。使われていれば 8766, 8767, … の空きを自動で使う
KWIC_LOG = OUT/'kwic_server.log'
KWIC_PID = OUT/'kwic_server.pid'

def port_open(port, host='127.0.0.1', timeout=0.4):
    try:
        with socket.create_connection((host, port), timeout):
            return True
    except OSError:
        return False

def whoami(port):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/api/whoami', timeout=2) as r:
            return json.loads(r.read().decode('utf-8'))
    except Exception:                                        # noqa: BLE001
        return {}

def mine(port):
    """そのポートのサーバが「自分の，この索引の」KWIC かどうか。

    127.0.0.1 はマシンの中の全ユーザーに共通なので，前の人のサーバ
    （ログアウトせずに離れた人のもの）が残っていることがある。"""
    w = whoami(port)
    return (w.get('user') == getpass.getuser()
            and w.get('index') == os.path.realpath(KW))

# 自分のサーバが既に起動していればそれを使い，無ければ最初の空いているポートを使う
PORTS = range(PORT0, PORT0 + 20)
PORT = next((p for p in PORTS if port_open(p) and mine(p)), None)
REUSE = PORT is not None
if REUSE:
    # **教材を更新した・索引を作り直したあとは，古いサーバを使い回さない。**
    # サーバはカーネルを再起動しても止まらないので，古いプログラム・古い索引のまま
    # 動き続ける（新しい画面と組み合わさると検索で表示が壊れる）
    from kwic_core import code_signature
    w = whoami(PORT)
    if w.get('sig') != code_signature(KW):
        print(f'[info ] 起動中のサーバ（ポート番号 {PORT}）は，教材の更新または'
              '索引の作り直しより前のもの。止めて起動し直す。')
        try:
            import signal
            os.kill(int(w['pid']), signal.SIGTERM)
        except (KeyError, ValueError, ProcessLookupError, PermissionError):
            # 古い版のサーバは pid を返さない。起動したときの記録から止める
            try:
                os.kill(int(KWIC_PID.read_text().split()[0]), signal.SIGTERM)
            except Exception:                                # noqa: BLE001
                pass
        for _ in range(20):
            if not port_open(PORT):
                break
            time.sleep(0.25)
        if port_open(PORT):
            print(f'[warn ] ポート番号 {PORT} のサーバを止められなかった。別のポートを使う。')
            PORT = None
        REUSE = False
if PORT is None:
    PORT = next((p for p in PORTS if not port_open(p)), None)

def show_link(port):
    display(HTML(f'<p style="font-size:1.05em">'
                 f'<a href="http://127.0.0.1:{port}/" target="_blank">'
                 f'KWIC コンコーダンサを開く（127.0.0.1:{port}）</a></p>'))
    print('リンクが開かないときは，ブラウザに '
          f'http://127.0.0.1:{port}/ を直接入れること。')

if REUSE:
    # 前に起動した自分のもの（カーネル再起動の前のものを含む）をそのまま使う
    print(f'[info ] すでに起動している（ポート番号 {PORT}）。止めるには下のセル。')
    show_link(PORT)
elif not (KW/'kwic_index.json').exists():
    print(f'[NG  ] 索引が無い: {KW}')
    print('       上の「索引を作る」セルを先に実行すること。')
elif PORT is None:
    print(f'[NG  ] ポート番号 {PORTS.start}–{PORTS.stop - 1} がすべて使われている。')
    print('       前の人のサーバが残っている可能性がある。いったんログアウトして'
          'ログインし直すか，TA に相談すること。')
else:
    if PORT != PORT0:
        print(f'[info ] ポート番号 {PORT0} は別のもの（前の人のサーバなど）が使っているので，'
              f'ポート番号 {PORT} を使う。')
    # -u: 出力を溜めずにログへ書く（溜めると動いていてもログが空に見える）
    # start_new_session: カーネルの中断・再起動に巻き込まれないようにする
    with open(KWIC_LOG, 'w', encoding='utf-8') as _log:
        KWIC_PROC = subprocess.Popen(
            [sys.executable, '-u', str(ROOT/'scripts'/'16_kwic_server.py'),
             '--index', str(KW), '--port', str(PORT)],
            stdout=_log, stderr=subprocess.STDOUT, text=True,
            start_new_session=True)
    KWIC_PID.write_text(f'{KWIC_PROC.pid} {PORT}\n', encoding='utf-8')
    t0 = time.time()
    ok = False
    while time.time() - t0 < 90:
        if KWIC_PROC.poll() is not None:
            break                       # エラーで止まった
        if port_open(PORT):
            ok = True
            break
        time.sleep(0.5)
        if int(time.time() - t0) and int(time.time() - t0) % 10 == 0:
            print(f'       …索引を読んでいる（{time.time() - t0:.0f} 秒）')
    if not ok:
        print('[NG  ] サーバが起動しなかった。ログの末尾:')
        print('-' * 60)
        print(KWIC_LOG.read_text(encoding='utf-8')[-2000:] or '（ログが空）')
        print('-' * 60)
        print('よくある原因: 索引が無い／ポートが使われている／'
              'Python が別の環境（sys.executable を確かめる）。')
        print(f'       sys.executable = {sys.executable}')
    else:
        print(f'[ok  ] 起動した（PID {KWIC_PROC.pid}・{time.time() - t0:.1f} 秒）'
              f'／ログ {KWIC_LOG}')
        show_link(PORT)

In [ ]:
# ---- 止める（使い終わったら STOP_KWIC = True にして実行する）----------
# 既定は False。上から順にセルを実行したとき，起動した直後に止めないため。
STOP_KWIC = False

import os, signal
KWIC_PID = OUT/'kwic_server.pid'
if not STOP_KWIC:
    print('[info ] 止めない（STOP_KWIC = False）。画面は起動したまま使える。')
    print('       使い終わったら STOP_KWIC = True にしてこのセルを実行する。')
else:
    pid = None
    proc = globals().get('KWIC_PROC')
    if proc is not None and proc.poll() is None:
        pid = proc.pid
    elif KWIC_PID.exists():                 # カーネルを再起動したあと
        pid = int(KWIC_PID.read_text().split()[0])
    if pid is None:
        print('[info ] 起動していない')
    else:
        try:
            os.kill(pid, signal.SIGTERM)
            print(f'[ok  ] 止めた（PID {pid}）')
        except ProcessLookupError:
            print('[info ] すでに止まっていた')
        KWIC_PID.unlink(missing_ok=True)

### ノートブックの中でも引ける

画面を使わずに，**表として**用例を出すこともできる（リポートに貼るとき・
図と並べて見たいとき）。`show()` に渡せる表が返る。

In [ ]:
# ---- ノートブックの中で KWIC を引く ----------------------------------
from kwic_core import KwicIndex, QueryError
kwi = KwicIndex(KW)
p = kwi.prov
print(f'索引: 辞書 {p["dictionary"]} {p["dictionary_version"]}／'
      f'{p["works"]} 作品・{p["tokens"]:,} 形態素／作成 {p["built_at"]}')

QUERY, STREAM = '汽車', 'lemma'     # 'lemma'（語彙素）か 'surface'（表層形）
try:
    res = kwi.search(QUERY, stream=STREAM, context=7, limit=15, sort='year')
    show(kwi.to_frame(res),
         caption=f'「{QUERY}」の用例（{"語彙素" if STREAM == "lemma" else "表層形"}・'
                 f'全 {res["total"]:,} 件のうち {res["shown"]} 件・初出年順）',
         align={'左文脈': 'right', 'キーワード': 'center'})
    show(pd.DataFrame(res['by_band']).rename(columns={
            'label': '時代区分', 'hits': 'ヒット', 'tokens': '形態素',
            'per_10k': '1万語あたり'})[['時代区分', 'ヒット', '形態素', '1万語あたり']],
         caption='時代区分ごとの分布（**生のヒット数で時代を比べてはいけない**）',
         fmt={'ヒット': '{:,.0f}', '形態素': '{:,.0f}', '1万語あたり': '{:.3f}'})
except QueryError as e:
    # **黙って0件にしない。** 語彙に無い語形のときは理由を出す
    print(f'[検索式] {e}')

### 演習 — 文語と口語の助動詞を**用例で**確かめる

Step 1 で `bungo_per10k`（文語助動詞の率）から文体を分類した。あれは数である。
**本当に文語の助動詞なのかを，用例で確かめる。**

1. 語彙素で `なり|けり|べし|ず` を引き，時代区分ごとの分布を見てみよう。
   数の上で減っているとして，**残っている用例はどういう文脈か**。
2. 同じ語を**表層形**で引き直し，何が変わるかを述べてみよう
   （「なり」は「なる」「なれ」と別の語になる。どちらの数字で議論すべきか）。
3. `L:なり /名詞` のように**列を混ぜた連なり**を作り，
   「なり」が助動詞ではなく名詞「成り」と解析されている例を探してみよう。
   **辞書の誤解析は，用例を読まなければ見つからない。**
4. 自分が Step 5 以降で追う語を5つ選び，それぞれ用例を10件読んでから
   仮説を書いてみよう。**数える前に読む**のが順序である。
5. 語彙素で `言う` を引き，異綴形の内訳を作家ごと・作品ごとに見てみよう。
   表記の違いは作家の選択か，底本・入力者の方針か。**作品ごとの内訳**と
   広げた本文を根拠に判断してみよう。
6. `汽車` の共起語を，品詞を「内容語（自立語）だけ」にして，LogDice・MI・
   ΔP（検索語→共起語）・ΔP（共起語→検索語）で上位を比べてみよう。
   指標によって上位に来る語がどう変わるか，その理由を指標の式から説明してみよう。

## 6. 検証 — 何が消え，何が残るのが正しいか

Step 1 で見た3つの重大な欠陥（重複・外字欠落・奥付混入）は消えているはずである。
ただし **FATAL 0 を目標にしてはいけない。** 検証スクリプトが出す警告には，
直すべきものと，直してはいけないものがある。

| 検査 | 出たら | 対処 |
|---|---|---|
| `duplicate` | 同一本文のファイルがある | **直す**。マニフェストを見直して取り直す |
| `gaiji_unresolved` (`〓`) | 実体が分からない外字がある | 注記を読んで対応字を探す。無ければそのまま。**件数と内訳を報告に書く** |
| `gaiji_marker` (`※`) | ※ が残っている | **判定する**。底本の記号（編者注・伏字）なら本文であって欠陥ではない |
| `colophon` | 奥付が本文に混入 | **直す**。`03` の本文抽出を見直す |
| `odoriji` | 踊り字が展開されていない | **直す**。`04` の設定を見直す |
| `length_skew` | 作品長の開きが大きい | 直さない。`--max-chunks` で吸収する（次節） |
| `too_short` | 本文が 3,000 字未満 | **確認する**。取得の失敗（本文抽出の打ち切り）か，本当に短い作品か |
| `length_skew_tokens` | 語数の開きが大きい | チャンク分割の判断はこちら。文字数ではなく語数で見る |

`gaiji_marker` を機械的に「欠陥」と数えると，海野十三の日記体作品の
編者注 `（※マリアナ基地からの…）` 100 箇所が誤って欠陥に数えられる。
**検証結果は読むものであって，0 にするものではない。**

In [ ]:
if need(PLAIN/'full', 'このステップの 04_normalise.py のセルを先に実行すること'):
    run_script('99_validate.py', '--corpus', PLAIN/'full',
               '--meta', META,
               '--tokenise-report', TOK/'tokenise_report.csv',
               '--out', OUT/'Step3_validation.csv')

## 7. このステップの課題

次の設問への答えを，テンプレート `my_work/results/Step3_report.md` に書いて提出する（**全体で600–1000字程度**。図表と「再現のための情報」は字数に含めない）。

- **提出先**：Zulip（dh-uosaka.zulipchat.com）の非公開チャネル **2026年度テクスト分析論B** ＞ トピック **Step 3**
- テンプレートの中身をメッセージに貼り付け，図（SVG）・表（CSV）は**同じメッセージに添付**する（1人1通）
- 図は番号で言及し（図1），**図を見なくても論旨が追えるように**書く（SVG は Zulip で表示されないことがある）
- 再提出は元の投稿を直さず，同じトピックに新しく投稿する（手順書 §5.3）

1. 未知語率が高い上位3ファイルについて，**理由を特定**し，対処案を書くこと。
2. `lemma-policy` を `surface` と `mixed` で切り替え，
   同一作品の異なり語数がどれだけ変わるかを報告すること。
3. `--chunk` を 1000 / 2000 / 5000 と変え，チャンク数の分布がどう変わるか図示すこと。
   トピックモデルにとってどれが望ましいか，理由とともに述べること。
   **`--min-chunk` の既定は `--chunk` の半分**なので，`--chunk` を大きくすると
   1チャンクに満たない作品が増え，**コーパスから除外される**。
   `works_too_short.csv` を見て，条件ごとに何点除外されたかも報告すること。
4. 検証ログを読み，ノートブックの表にしたがって**各警告を「直す／直さない」に分類**し，
   直さないものについてはその理由を1行で書くこと。FATAL 0 は目標ではない。
5. `narration_excludes_embedded` を `true` にして `04` を再実行し，
   『こころ』の `narration` の字数がどれだけ減るかを示すこと。
   どちらの設定が自分の問いに適しているかを述べること。
6. **（発展）辞書を2つ用意して `12_dict_compare.py` を実行し**，
   未知語率・平均語長・境界一致率を層別に出すこと。
   `dict_disagreements.csv` から食い違いを**3例選んで原文と照らし**，
   どちらの切り方が自分の問いに適しているかを述べること。
   **未知語率が低いほうを無条件に選んではいけない**理由も書くこと。

### このステップの到達点（次へ進む条件）

- `data/plain/{full,narration,speech,embedded}/` がそろっている
- `data/tokens_*/` と `tokenise_report.csv` が出ており，未知語率を言える
- **`tokenise_provenance.json` に本番の辞書（`unidic-novel`）が記録されている**
- `data/datasets/chunks_index.csv` があり，時代別チャンク数の偏りを把握している
- 検証の警告を一つずつ読み，直すもの・直さないものを仕分けてある
